# 머신러닝-3_평가지표·로지스틱

평가지표 노트북 + 로지스틱 회귀 노트북(시그모이드 · 확률 · 임곗값)을 **중복 없이 6개 대단원 · 16개 섹션**으로 재구성하고, **7장 종합 실습**에서 타이타닉을 분할 → 파이프라인 → 교차검증 → 임곗값 선택 → 최종 test 1회까지 처음부터 끝까지 연결. 두 원본에서 겹치는 임곗값 · 정밀도/재현율은 한 번만 다루고, 분할 · 교차검증은 [머신러닝-1](머신러닝-1_데이터분할·교차검증%28완성본%29.ipynb), 전처리 · 파이프라인은 [머신러닝-2](머신러닝-2_전처리·누수·파이프라인%28완성본%29.ipynb)를 그대로 가져다 씀.

## 이 노트북을 끝내면 답할 수 있어야 하는 질문

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">대단원</th>
<th style="text-align: left;">질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>1. 정확도의 함정</strong></td>
<td style="text-align: left;">정확도가 같은 두 모델은 같은 실력인가? 불균형 데이터에서 정확도는 왜 착시를 만드는가?</td>
</tr>
<tr>
<td style="text-align: left;"><strong>2. 혼동행렬과 지표</strong></td>
<td style="text-align: left;">실수를 어떻게 나누며, 정확도 · 정밀도 · 재현율 · F1은 각각 어떤 실수를 보는가?</td>
</tr>
<tr>
<td style="text-align: left;"><strong>3. 로지스틱 회귀와 확률</strong></td>
<td style="text-align: left;">점수는 어떻게 확률이 되고, <code>predict</code>와 <code>predict_proba</code>는 무엇이 다르며, 확률을 믿어도 되는가?</td>
</tr>
<tr>
<td style="text-align: left;"><strong>4. 임곗값</strong></td>
<td style="text-align: left;">판정 기준을 바꾸면 무엇이 변하고, 업무 기준은 어느 데이터로 고르는가?</td>
</tr>
<tr>
<td style="text-align: left;"><strong>5. 곡선 지표</strong></td>
<td style="text-align: left;">ROC-AUC와 PR-AUC는 무엇을 재며, 불균형에서는 어느 쪽을 보는가?</td>
</tr>
<tr>
<td style="text-align: left;"><strong>6. 딥러닝 연결</strong></td>
<td style="text-align: left;">시그모이드 출력층 · BCE 손실 · 불균형 가중치는 이 장의 무엇과 같은가?</td>
</tr>
<tr>
<td style="text-align: left;"><strong>7. 종합 실습</strong></td>
<td style="text-align: left;">타이타닉을 분할부터 최종 test 1회까지 규칙대로 끝내려면?</td>
</tr>
</tbody>
</table>

## 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

**섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 예측 정답 · 결과 해설 → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리

**코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 실행 흐름 · 문법 구조

**셀 실행** · 위에서부터 차례로 (앞 섹션의 변수를 이어서 씀), 노트북과 같은 폴더의 `train.csv`(타이타닉)를 읽음, 6장은 PyTorch 사용

**앞 권에서 가져다 쓰는 것** · 3분할 · test 1회 · `StratifiedKFold` · `cross_validate`(머신러닝-1), Cabin1 · `ColumnTransformer` · `SimpleImputer` · `OneHotEncoder` · Pipeline째 학습(머신러닝-2) → 이 노트북에서는 설명 없이 사용

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">섹션</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">04 평가지표 Part 1 셀 4 ~ 8 + 4-1 로지스틱 셀 19 ~ 22</td>
<td style="text-align: left;">남자면 사망 규칙 77% · 로지스틱과 같은 정확도</td>
<td style="text-align: left;">1-1 (<strong>통합</strong>: 같은 장면이라 4-1의 valid 숫자로 1회)</td>
</tr>
<tr>
<td style="text-align: left;">04 Part 2 셀 9 ~ 15</td>
<td style="text-align: left;">digits 7 vs 나머지 · 전부 0 예측</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">04 Part 3 · 4 셀 16 ~ 27</td>
<td style="text-align: left;">Positive 정하기 · 혼동행렬 · 정밀도 · 재현율 · F1 · macro · weighted</td>
<td style="text-align: left;">2-1 ~ 2-3</td>
</tr>
<tr>
<td style="text-align: left;">4-1 Part 1 셀 3 ~ 5</td>
<td style="text-align: left;">시그모이드</td>
<td style="text-align: left;">3-1 (plotly 그래프 → 표)</td>
</tr>
<tr>
<td style="text-align: left;">4-1 Part 3 셀 10 ~ 18 · 35 ~ 43 · 48 ~ 52</td>
<td style="text-align: left;">predict_proba · 반복 횟수 · 교차 엔트로피 · 확신하고 틀림 · z와 거리 · 계수 · 특성 조합 AUC</td>
<td style="text-align: left;">3-2 ~ 3-4 (특성 조합 AUC · 성별 × 아이 → 8-2 문제)</td>
</tr>
<tr>
<td style="text-align: left;">04 Part 5 셀 28 ~ 41 + 4-1 Part 4 셀 23 ~ 31</td>
<td style="text-align: left;">임곗값 · 선택 규칙 · validation 선택 · test 1회 · 줄다리기 예외</td>
<td style="text-align: left;">4-1 ~ 4-3 (<strong>통합</strong>: 정의는 1회, digits는 선택 절차, 타이타닉은 움직임)</td>
</tr>
<tr>
<td style="text-align: left;">04 Part 6 셀 42 ~ 51</td>
<td style="text-align: left;">ROC · AUC · PR · AP</td>
<td style="text-align: left;">5-1 · 5-2 (불균형에서 AP만 떨어지는 실험 추가)</td>
</tr>
<tr>
<td style="text-align: left;">4-1 Part 6 셀 55 ~ 57</td>
<td style="text-align: left;">수능 보러 가기 (test 1회)</td>
<td style="text-align: left;">7장 종합 실습</td>
</tr>
<tr>
<td style="text-align: left;">(추가)</td>
<td style="text-align: left;">이진 분류 출력층 · BCE · 불균형 가중치</td>
<td style="text-align: left;">6-1 · 6-2</td>
</tr>
</tbody>
</table>

**제외** · 4-1 셀 53(정수 라벨 vs 원-핫)은 머신러닝-2 1-2와 같은 내용이라 생략, 4-2 KNN · 앙상블 · XGBoost는 범위 밖. plotly 그래프는 같은 숫자를 표로 출력. 원본 파일은 수정하지 않음.

## 전체 지도

### 단원 연결 · 왜 이 순서로 배우는가

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">순서</th>
<th style="text-align: left;">배우는 이유</th>
<th style="text-align: left;">대단원</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">머신러닝-1 · 2에서 쓴 정확도가 무엇을 숨기는지 본다</td>
<td style="text-align: left;">정확도의 함정</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">실수를 종류별로 나눠 세는 지표를 만든다</td>
<td style="text-align: left;">혼동행렬과 지표</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">0/1 판정 앞에 있던 확률을 연다</td>
<td style="text-align: left;">로지스틱 회귀와 확률</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">확률을 0/1로 바꾸는 기준을 업무에 맞게 고른다</td>
<td style="text-align: left;">임곗값</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">기준을 정하기 전, 모든 기준에서의 실력을 한 숫자로 본다</td>
<td style="text-align: left;">곡선 지표</td>
</tr>
<tr>
<td style="text-align: center;">6</td>
<td style="text-align: left;">같은 구조가 신경망 출력층 · 손실로 이어진다</td>
<td style="text-align: left;">딥러닝 연결</td>
</tr>
<tr>
<td style="text-align: center;">7</td>
<td style="text-align: left;">머신러닝-1 · 2 · 3을 타이타닉 하나에 모두 적용한다</td>
<td style="text-align: left;">종합 실습</td>
</tr>
</tbody>
</table>

### ML/DL 공통 파이프라인 · 각 대단원이 쓰이는 곳

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰이는 장</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">② 나누기</td>
<td style="text-align: left;">train · valid · test (머신러닝-1)</td>
<td style="text-align: left;">0장 · 4-2 · 7장</td>
</tr>
<tr>
<td style="text-align: left;">③ 전처리</td>
<td style="text-align: left;">Pipeline (머신러닝-2)</td>
<td style="text-align: left;">0장 · 7장</td>
</tr>
<tr>
<td style="text-align: left;">④ 모델</td>
<td style="text-align: left;">로지스틱 = 시그모이드 출력층</td>
<td style="text-align: left;"><strong>3장</strong> · <strong>6장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑤ 학습</td>
<td style="text-align: left;">교차 엔트로피 · 불균형 가중치</td>
<td style="text-align: left;">3-3 · <strong>6장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑥ 평가 · 선택</td>
<td style="text-align: left;">지표 고르기 · 임곗값 선택 (validation)</td>
<td style="text-align: left;"><strong>1장 ~ 5장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑦ 최종 보고</td>
<td style="text-align: left;">고른 기준으로 test 1회</td>
<td style="text-align: left;">4-2 · <strong>7장</strong></td>
</tr>
</tbody>
</table>

**굵은 장** · 이 노트북의 핵심 = ⑥ 무엇으로 평가하고 무엇을 validation에서 고르는가

---
## 0. 준비

모든 장에서 쓰는 라이브러리 · 타이타닉 3분할 · 전처리 함수 · digits를 준비. 타이타닉 정리와 분할은 머신러닝-2와 **완전히 같아서** valid · test 숫자를 그대로 이어서 비교할 수 있음.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">데이터 · 함수</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">처음 나오는 곳</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>X_train</code> · <code>X_valid</code> · <code>X_test</code></td>
<td style="text-align: left;">타이타닉 534 · 178 · 179명 (test는 7장에서 처음 씀)</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;"><code>make_lr(cols, C)</code></td>
<td style="text-align: left;">머신러닝-2 4-3의 전처리 + 로지스틱 (쓸 열만 골라서)</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;"><code>X_dig</code> · <code>y7</code></td>
<td style="text-align: left;">digits 1797장 · 숫자 7이면 1 (10%)</td>
<td style="text-align: left;">0</td>
</tr>
<tr>
<td style="text-align: left;"><code>X7_tr</code> · <code>X7_te</code></td>
<td style="text-align: left;">digits 8 : 2 (원본 Part 1 ~ 4)</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;"><code>X7_fit</code> · <code>X7_val</code> · <code>X7_test</code></td>
<td style="text-align: left;">digits 3분할 (원본 Part 5 ~ 6, 기준 선택용)</td>
<td style="text-align: left;">4-2</td>
</tr>
</tbody>
</table>

In [1]:
# 라이브러리 불러오기
import sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score)

SEED = 42
print("sklearn:", sklearn.__version__)


# 타이타닉: 머신러닝-2와 같은 정리 · 같은 3분할 (test는 7장까지 봉인)
titanic = pd.read_csv("train.csv")
X_t = titanic.drop(columns="Survived").copy()
X_t["Cabin1"] = X_t["Cabin"].fillna("N").str[:1]
X_t = X_t.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
y_t = titanic["Survived"]
X_dev, X_test, y_dev, y_test = train_test_split(
    X_t, y_t, test_size=0.2, stratify=y_t, random_state=SEED)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)
print(f"train {len(X_train)} | valid {len(X_valid)} | test {len(X_test)}")


# 머신러닝-2 4-3과 같은 전처리 + 로지스틱 묶음 (쓸 열만 골라서)
NUM = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CAT = ["Sex", "Embarked", "Cabin1"]


def make_lr(cols=NUM + CAT, C=1.0):
    num = Pipeline([("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())])
    cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    prep = ColumnTransformer([
        ("num", num, [c for c in cols if c in NUM]),
        ("cat", cat, [c for c in cols if c in CAT])])
    return Pipeline([("prep", prep),
                     ("model", LogisticRegression(max_iter=5000, C=C))])


# digits: 숫자 7이면 1 (Positive), 나머지 0 → 불균형 10 : 90
digits = load_digits()
X_dig, y7 = digits.data, (digits.target == 7).astype(int)
print("digits:", X_dig.shape, "| 7의 비율:", round(y7.mean(), 3))

sklearn: 1.9.0
train 534 | valid 178 | test 179


digits: (1797, 64) | 7의 비율: 0.1


---
## 1. 정확도의 함정

> **연결** · 머신러닝-1 · 2에서는 정확도 하나로 모델을 비교했다 → 정확도는 맞힌 개수만 세서, 규칙 한 줄과 학습한 모델을 구별하지 못하고 불균형에서는 착시를 만듦

**흐름 위치** · ⑥ 평가 — 기준선과 비교, Positive 정하기

### 1-1. `A` 정확도가 같은 두 모델 · `accuracy_score` <sub>원본 04 Part 1 셀 4 ~ 8 · 4-1 로지스틱 셀 19 ~ 22</sub>

> 🎯 **특성 8개로 학습한 로지스틱과 '남자면 사망' 규칙 한 줄의 정확도가 같다면, 두 모델은 같은 실력인가?**

💡 **쉽게 말하면** 정확도는 **맞힌 문제 수만 적힌 성적표**. 같은 점수라도 어떤 문제를 틀렸는지, 얼마나 자신 있게 풀었는지는 적혀 있지 않음.

**실행 흐름** · valid에서 ① 전부 사망 ② 남자면 사망 ③ 로지스틱 정확도 → ②와 ③의 예측이 다른 승객 수

**먼저 예측** · 로지스틱(특성 8개 · 전처리 19열)의 정확도는 규칙 한 줄보다 얼마나 높을까? — 답은 코드 실행 결과 아래에

In [2]:
# 비교할 예측 3가지 (모두 valid 178명)
lr_all = make_lr().fit(X_train, y_train)  # 특성 8개 로지스틱
preds = {
    "① 전부 사망": np.zeros(len(y_valid), dtype=int),  # 다수 클래스
    "② 남자면 사망": (X_valid["Sex"] == "female").astype(int).to_numpy(),
    "③ 로지스틱": lr_all.predict(X_valid),
}


# 정확도만 보면?
for name, p in preds.items():
    print(f"{name:<8} 정확도 {accuracy_score(y_valid, p):.3f}")


# 같은 정확도인데 예측도 같을까?
diff = (preds["② 남자면 사망"] != preds["③ 로지스틱"]).sum()
print("②와 ③의 예측이 다른 승객:", diff, "명")

① 전부 사망  정확도 0.618
② 남자면 사망 정확도 0.798
③ 로지스틱   정확도 0.798
②와 ③의 예측이 다른 승객: 22 명


**예측 정답** · 같음 — 둘 다 `0.798` (원본 4-1의 '충격' 장면과 같은 숫자)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>① 전부 사망  정확도 0.618</code></td>
<td style="text-align: left;">다수 클래스(사망) 비율 = 아무것도 안 배운 기준선</td>
</tr>
<tr>
<td style="text-align: left;"><code>② 남자면 사망 정확도 0.798</code></td>
<td style="text-align: left;">학습 없는 규칙 한 줄</td>
</tr>
<tr>
<td style="text-align: left;"><code>③ 로지스틱   정확도 0.798</code></td>
<td style="text-align: left;">정확도만으로는 ②와 구별 불가</td>
</tr>
<tr>
<td style="text-align: left;"><code>②와 ③의 예측이 다른 승객: 22 명</code></td>
<td style="text-align: left;">같은 점수인데 맞힌 사람이 다름 → 3장 확률 · 5장 AUC에서 차이가 드러남</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도 하나만</td>
<td style="text-align: left;">맞힌 <strong>개수</strong>만 셈 → 무엇을 틀렸는지 · 얼마나 확신했는지 모름</td>
<td style="text-align: left;">클래스가 고르고 실수 비용이 같을 때의 출발점</td>
</tr>
<tr>
<td style="text-align: left;">기준선(다수 클래스 · 단순 규칙)과 비교</td>
<td style="text-align: left;">모델이 <strong>규칙 한 줄보다 나은지</strong> 확인</td>
<td style="text-align: left;">모든 평가의 첫 줄 (기본 선택)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · 정확도는 반드시 기준선(다수 클래스 · 단순 규칙)과 나란히 — 숫자 하나만 보고 "80%면 좋다" 판단 금지

> ✅ **한 줄 정리** · 정확도가 같아도 무엇을 맞히고 틀렸는지는 다르므로, 기준선과 함께 다른 지표로 확인한다.

**딥러닝 코드에서는** · 딥러닝 분류도 첫 epoch 전에 '전부 다수 클래스' 정확도를 계산해 두면 학습이 진짜 배우고 있는지 판단 가능

### 1-2. `A` 불균형 데이터의 착시 · `zero_division` <sub>원본 04 Part 2 셀 9 ~ 15</sub>

> 🎯 **Positive가 10%뿐인 데이터에서 전부 Negative로 찍으면 정확도와 재현율은 각각 얼마인가?**

💡 **쉽게 말하면** 100명 중 환자 10명인 병원에서 **모두 건강하다고 말하는 의사**도 90점. 그런데 환자는 한 명도 못 찾음.

**실행 흐름** · digits 숫자 7(10%) vs 나머지 → 전부 0 예측 → 정확도 · 정밀도 · 재현율 · 찾은 7의 수

**먼저 예측** · 아무것도 안 배운 '전부 0' 모델의 정확도는? — 답은 코드 실행 결과 아래에

In [3]:
# 원본 셀: digits 8 : 2 분할 (random_state=11)
X7_tr, X7_te, y7_tr, y7_te = train_test_split(
    X_dig, y7, test_size=0.2, stratify=y7, random_state=11)
print("test 7의 비율:", y7_te.mean().round(3), "|", len(y7_te), "장")


# 모두 0(7 아님)으로 찍는 가짜 모델
fake = np.zeros(len(y7_te), dtype=int)
print(f"정확도  : {accuracy_score(y7_te, fake):.3f}")
print("정밀도  :", precision_score(y7_te, fake, zero_division=0))  # 양성 예측 0개
print("재현율  :", recall_score(y7_te, fake, zero_division=0))
print("찾은 7  :", int(fake[y7_te == 1].sum()), "/", int(y7_te.sum()), "장")

test 7의 비율: 0.1 | 360 장
정확도  : 0.900
정밀도  : 0.0
재현율  : 0.0
찾은 7  : 0 / 36 장


**예측 정답** · `0.900` — 그런데 재현율 `0.0`, 7을 하나도 못 찾음

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>test 7의 비율: 0.1 &#124; 360 장</code></td>
<td style="text-align: left;">Positive(7)는 36장뿐, 나머지가 Negative</td>
</tr>
<tr>
<td style="text-align: left;"><code>정확도  : 0.900</code></td>
<td style="text-align: left;">다수 클래스 비율 그대로</td>
</tr>
<tr>
<td style="text-align: left;"><code>정밀도  : 0.0</code></td>
<td style="text-align: left;">Positive 예측이 0개 → 0으로 처리 (<code>zero_division=0</code>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>재현율  : 0.0</code></td>
<td style="text-align: left;">7을 하나도 못 찾음</td>
</tr>
<tr>
<td style="text-align: left;"><code>찾은 7  : 0 / 36 장</code></td>
<td style="text-align: left;">정확도가 감춘 사실</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도</td>
<td style="text-align: left;">다수 클래스만 맞혀도 높음 (0.900)</td>
<td style="text-align: left;">균형 데이터</td>
</tr>
<tr>
<td style="text-align: left;">재현율 · 정밀도 · F1</td>
<td style="text-align: left;"><strong>Positive</strong>(관심 대상) 기준 → 하나도 못 찾으면 0</td>
<td style="text-align: left;">불균형 데이터 (사기 · 질병 · 불량)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · 지표를 보기 전에 **Positive = 관심 대상**을 먼저 적기 (여기서는 숫자 7) — Positive를 바꾸면 정밀도 · 재현율 값도 바뀜

> ✅ **한 줄 정리** · 불균형 데이터에서 정확도는 다수 클래스만 맞혀도 높으므로, Positive를 정하고 재현율 · 정밀도로 본다.

**딥러닝 코드에서는** · 불량 검출 · 이상 탐지 · 질병 영상 프로젝트는 대부분 불균형 → 학습 로그에 정확도만 찍지 말고 재현율 · F1을 함께 기록

#### 1장 정리 · 소속별 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>accuracy_score(y, pred)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">정확도</td>
</tr>
<tr>
<td style="text-align: left;"><code>precision_score(…, zero_division=0)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">양성 예측이 없을 때 0으로</td>
</tr>
<tr>
<td style="text-align: left;"><code>load_digits()</code></td>
<td style="text-align: left;">sklearn.datasets</td>
<td style="text-align: left;">8 × 8 숫자 손글씨 1797장</td>
</tr>
<tr>
<td style="text-align: left;"><code>(y == 7).astype(int)</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">관심 대상 = 1 (Positive)</td>
</tr>
</tbody>
</table>

---
## 2. 혼동행렬과 지표

> **연결** · 1장에서 정확도가 무엇을 숨기는지 봤다 → 맞음 · 틀림을 4칸으로 나누면 숨은 실수가 보이고, 그 칸들로 실수 종류별 지표를 만듦

**흐름 위치** · ⑥ 평가 — 오탐 · 놓침을 따로 세기

### 2-1. `A` 실수를 4칸으로 나누기 · `confusion_matrix` <sub>원본 04 Part 3 셀 16 ~ 20</sub>

> 🎯 **모델의 맞음과 틀림을 어떤 4칸으로 나누며, 두 모델의 혼동행렬은 어떻게 다른가?**

💡 **쉽게 말하면** 혼동행렬은 **실제 × 예측 2 × 2 표**. 맞힌 칸(TN · TP)과 두 종류의 실수 칸 — 없는 걸 있다고 함(FP, 오탐) · 있는 걸 놓침(FN) — 으로 나눔.

**실행 흐름** · 가짜(전부 0) · 원본 기준선 로지스틱 → confusion_matrix → ravel로 TN FP FN TP

**먼저 예측** · 로지스틱이 놓친 7(FN)은 몇 장일까? — 답은 코드 실행 결과 아래에

In [4]:
# 원본 기준선: 약하게 만들지 않은 로지스틱 (digits 64픽셀 그대로)
lr7 = LogisticRegression(max_iter=5000).fit(X7_tr, y7_tr)
pred7 = lr7.predict(X7_te)


# 혼동행렬: 행 = 실제 0 · 1, 열 = 예측 0 · 1 → ravel 순서 TN FP FN TP
for name, p in [("가짜(전부 0)", fake), ("로지스틱", pred7)]:
    cm = confusion_matrix(y7_te, p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f"{name}: TN {tn} · FP {fp} · FN {fn} · TP {tp}"
          f" | 표 {cm.tolist()}")

가짜(전부 0): TN 324 · FP 0 · FN 36 · TP 0 | 표 [[324, 0], [36, 0]]
로지스틱: TN 324 · FP 0 · FN 2 · TP 34 | 표 [[324, 0], [2, 34]]


**예측 정답** · `2`장 — TN 324 · FP 0 · FN 2 · TP 34

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>가짜(전부 0): TN 324 · FP 0 · FN 36 · TP 0 &#124; 표 [[324, 0], [36, 0]]</code></td>
<td style="text-align: left;">7 36장을 전부 놓침 (FN 36)</td>
</tr>
<tr>
<td style="text-align: left;"><code>로지스틱: TN 324 · FP 0 · FN 2 · TP 34 &#124; 표 [[324, 0], [2, 34]]</code></td>
<td style="text-align: left;">오탐 0, 놓침 2 → 실수의 종류가 보임</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>confusion_matrix(y, pred)</code></td>
<td style="text-align: left;">4칸 개수 그대로 (행 = 실제, 열 = 예측)</td>
<td style="text-align: left;">어떤 실수가 몇 개인지 볼 때 (항상 먼저)</td>
</tr>
<tr>
<td style="text-align: left;"><code>.ravel()</code> → <code>tn, fp, fn, tp</code></td>
<td style="text-align: left;">4칸을 이름으로 받음 (순서 고정)</td>
<td style="text-align: left;">공식에 넣어 지표를 직접 계산할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · sklearn 표는 **행 = 실제, 열 = 예측**, `ravel()` 순서는 TN · FP · FN · TP — 책 · 블로그마다 행렬 방향이 달라 순서부터 확인

> ✅ **한 줄 정리** · 혼동행렬로 맞음을 TN · TP, 실수를 FP(오탐) · FN(놓침)으로 나눠 무엇을 틀렸는지 먼저 본다.

**딥러닝 코드에서는** · 다중 분류(이미지 10종)도 `confusion_matrix(y, logits.argmax(1))` → 어떤 클래스끼리 헷갈리는지 (고양이 ↔ 개) 확인

### 2-2. `A` 정밀도 · 재현율 · F1 · `precision_score` · `recall_score` · `f1_score` <sub>원본 04 Part 4 셀 21 ~ 25</sub>

> 🎯 **정확도 · 정밀도 · 재현율 · F1은 혼동행렬의 어느 칸으로 계산하며, 각각 어떤 실수를 보는가?**

💡 **쉽게 말하면** 정밀도는 **'7이라고 외친 것 중 진짜 7'**(헛발질 검사), 재현율은 **'진짜 7 중 찾아낸 것'**(놓침 검사), F1은 둘 중 낮은 쪽에 끌려가는 평균.

**실행 흐름** · TN FP FN TP → 공식 4개 직접 계산 → sklearn 함수와 비교 → 가짜 · 로지스틱을 4개 지표로 나란히

**먼저 예측** · 로지스틱의 정밀도와 재현율 중 어느 쪽이 1.000일까? — 답은 코드 실행 결과 아래에

In [5]:
# 공식으로 직접 계산 (로지스틱의 TN 324 · FP 0 · FN 2 · TP 34)
tn, fp, fn, tp = confusion_matrix(y7_te, pred7).ravel()
acc = (tp + tn) / (tp + tn + fp + fn)
prec = tp / (tp + fp)  # 7이라고 한 것 중 진짜 7
rec = tp / (tp + fn)  # 진짜 7 중 찾은 것
f1 = 2 * prec * rec / (prec + rec)  # 조화평균
print(f"공식  : 정확도 {acc:.4f} · 정밀도 {prec:.4f}"
      f" · 재현율 {rec:.4f} · F1 {f1:.4f}")


# sklearn 함수와 비교 (모두 Positive = 1 기준)
print(f"sklearn: 정확도 {accuracy_score(y7_te, pred7):.4f}"
      f" · 정밀도 {precision_score(y7_te, pred7):.4f}"
      f" · 재현율 {recall_score(y7_te, pred7):.4f}"
      f" · F1 {f1_score(y7_te, pred7):.4f}")


# 같은 4개 지표로 두 모델 비교
for name, p in [("가짜(전부 0)", fake), ("로지스틱", pred7)]:
    scores = [accuracy_score(y7_te, p),
              precision_score(y7_te, p, zero_division=0),
              recall_score(y7_te, p, zero_division=0),
              f1_score(y7_te, p, zero_division=0)]
    print(f"{name:<8}", np.round(scores, 3))

공식  : 정확도 0.9944 · 정밀도 1.0000 · 재현율 0.9444 · F1 0.9714
sklearn: 정확도 0.9944 · 정밀도 1.0000 · 재현율 0.9444 · F1 0.9714
가짜(전부 0) [0.9 0.  0.  0. ]
로지스틱     [0.994 1.    0.944 0.971]


**예측 정답** · 정밀도 — FP가 0이라 `34 / 34 = 1.000`, 재현율은 `34 / 36 = 0.944`

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>공식  : 정확도 0.9944 · 정밀도 1.0000 · 재현율 0.9444 · F1 0.9714</code></td>
<td style="text-align: left;">TP · TN · FP · FN으로 직접</td>
</tr>
<tr>
<td style="text-align: left;"><code>sklearn: 정확도 0.9944 · 정밀도 1.0000 · 재현율 0.9444 · F1 0.9714</code></td>
<td style="text-align: left;">함수 결과와 같음</td>
</tr>
<tr>
<td style="text-align: left;"><code>가짜(전부 0) [0.9 0.  0.  0. ]</code></td>
<td style="text-align: left;">정확도만 높고 나머지 셋은 0</td>
</tr>
<tr>
<td style="text-align: left;"><code>로지스틱     [0.994 1.    0.944 0.971]</code></td>
<td style="text-align: left;">넷 다 높음 → 진짜 실력</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도 (TP + TN) / 전체</td>
<td style="text-align: left;">전체 중 맞힌 비율</td>
<td style="text-align: left;">클래스가 고를 때</td>
</tr>
<tr>
<td style="text-align: left;">정밀도 TP / (TP + FP)</td>
<td style="text-align: left;">Positive라고 한 것 중 진짜 → <strong>오탐(FP)</strong> 비용</td>
<td style="text-align: left;">스팸 차단 · 추천 (잘못 울리면 손해)</td>
</tr>
<tr>
<td style="text-align: left;">재현율 TP / (TP + FN)</td>
<td style="text-align: left;">진짜 Positive 중 찾은 것 → <strong>놓침(FN)</strong> 비용</td>
<td style="text-align: left;">암 진단 · 사기 탐지 (놓치면 손해)</td>
</tr>
<tr>
<td style="text-align: left;">F1 = 2PR / (P + R)</td>
<td style="text-align: left;">정밀도 · 재현율의 조화평균 → 한쪽이 0이면 0</td>
<td style="text-align: left;">두 실수 비용이 비슷할 때 하나의 숫자로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · `precision_score(정답, 예측)` 순서 — 바꾸면 정밀도와 재현율이 서로 뒤바뀜, Positive는 기본 `pos_label=1`

> ✅ **한 줄 정리** · 정밀도는 오탐(FP), 재현율은 놓침(FN)을 보는 지표이고, F1은 둘의 조화평균이며, 업무에서 더 비싼 실수로 우선 지표를 고른다.

**딥러닝 코드에서는** · 학습 루프의 평가 함수에 `precision_recall_fscore_support`를 넣어 epoch마다 F1을 기록하고 `best.pt` 기준으로 씀

### 2-3. `B` 클래스별 지표 평균 내기 · `classification_report` <sub>원본 04 셀 26 ~ 27</sub>

> 🎯 **클래스별로 나온 지표를 하나로 합치는 macro · weighted · balanced accuracy는 무엇이 다른가?**

💡 **쉽게 말하면** 반 평균을 낼 때 **반마다 1표씩**(macro)이냐, **학생 수만큼 표**(weighted)냐의 차이. 학생이 적은 반(소수 클래스)의 목소리는 macro에서 커짐.

**실행 흐름** · classification_report → macro · weighted 줄 비교 → balanced accuracy = 클래스별 재현율 평균

**먼저 예측** · macro 재현율과 balanced accuracy는 같을까? — 답은 코드 실행 결과 아래에

In [6]:
from sklearn.metrics import balanced_accuracy_score, classification_report


# 클래스별 지표 → 평균 내는 방식 3가지
print(classification_report(y7_te, pred7, digits=3))
print(f"balanced accuracy: {balanced_accuracy_score(y7_te, pred7):.4f}"
      f" (= 클래스별 재현율 평균 (1.000 + 0.944) / 2)")

              precision    recall  f1-score   support

           0      0.994     1.000     0.997       324
           1      1.000     0.944     0.971        36

    accuracy                          0.994       360
   macro avg      0.997     0.972     0.984       360
weighted avg      0.994     0.994     0.994       360



balanced accuracy: 0.9722 (= 클래스별 재현율 평균 (1.000 + 0.944) / 2)


**예측 정답** · 같음 — 둘 다 `0.972` (정의가 같음)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>1      1.000     0.944     0.971        36</code></td>
<td style="text-align: left;">숫자 7(Positive)의 정밀도 · 재현율 · F1 · 개수</td>
</tr>
<tr>
<td style="text-align: left;"><code>macro avg      0.997     0.972     0.984       360</code></td>
<td style="text-align: left;">두 클래스를 같은 비중으로</td>
</tr>
<tr>
<td style="text-align: left;"><code>weighted avg      0.994     0.994     0.994       360</code></td>
<td style="text-align: left;">324 : 36 비중 → 다수 클래스(0)에 가까움</td>
</tr>
<tr>
<td style="text-align: left;"><code>balanced accuracy: 0.9722</code></td>
<td style="text-align: left;">macro 재현율과 같은 값</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">macro 평균</td>
<td style="text-align: left;">클래스별 지표를 <strong>똑같은 비중</strong>으로 평균</td>
<td style="text-align: left;">소수 클래스도 같게 볼 때</td>
</tr>
<tr>
<td style="text-align: left;">weighted 평균</td>
<td style="text-align: left;">샘플 <strong>수만큼 가중</strong> → 다수 클래스에 가까움</td>
<td style="text-align: left;">실제 분포를 반영할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>balanced_accuracy</code></td>
<td style="text-align: left;">클래스별 재현율의 평균 (= macro 재현율)</td>
<td style="text-align: left;">불균형에서 정확도 대신</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · weighted 평균은 불균형에서 정확도처럼 높게 나옴 — 소수 클래스를 보려면 macro 또는 클래스별 줄

> ✅ **한 줄 정리** · 클래스별 지표는 macro(같은 비중) · weighted(샘플 수 비중)로 합치며, 소수 클래스가 중요하면 macro와 balanced accuracy를 본다.

**딥러닝 코드에서는** · 다중 분류 대회 지표 `macro F1`이 바로 이것 — `f1_score(y, pred, average="macro")`

#### 2장 정리 · 소속별 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>confusion_matrix(y, pred, labels=)</code> · <code>.ravel()</code></td>
<td style="text-align: left;">sklearn.metrics · ndarray</td>
<td style="text-align: left;">4칸 표 · TN FP FN TP</td>
</tr>
<tr>
<td style="text-align: left;"><code>precision_score</code> · <code>recall_score</code> · <code>f1_score</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">정밀도 · 재현율 · F1</td>
</tr>
<tr>
<td style="text-align: left;"><code>classification_report(digits=)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">클래스별 표 + macro · weighted</td>
</tr>
<tr>
<td style="text-align: left;"><code>balanced_accuracy_score</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">클래스별 재현율 평균</td>
</tr>
</tbody>
</table>

---
## 3. 로지스틱 회귀와 확률

> **연결** · 2장의 지표는 모두 0/1 판정으로 계산했다 → 그 판정 앞에는 확률이 있고, 확률은 점수 z와 시그모이드에서 나옴. 확률을 열어야 4장의 기준을 바꿀 수 있음

**흐름 위치** · ④ 모델 · ⑤ 학습 — 점수 → 확률 → 교차 엔트로피

### 3-1. `A` 점수를 확률로 · `sigmoid` <sub>원본 4-1 로지스틱 Part 1 셀 3 ~ 5</sub>

> 🎯 **로지스틱 회귀는 계산한 점수를 어떻게 0 ~ 1 확률로 바꾸며, 기준 0.5는 점수로는 어디인가?**

💡 **쉽게 말하면** 시그모이드는 **점수를 확률로 바꾸는 변환기**. 점수가 아무리 커도 1을 넘지 않고, 점수 0은 정확히 반반(0.5).

**실행 흐름** · 시그모이드 함수 정의 → z 5개 변환 → σ(0) · 대칭 성질 → 가운데 · 양끝에서 z를 1 올릴 때 확률 변화

**먼저 예측** · z를 0 → 1로 올릴 때와 4 → 5로 올릴 때 확률은 똑같이 오를까? — 답은 코드 실행 결과 아래에

In [7]:
# 시그모이드: 점수 z(-∞ ~ +∞) → 확률(0 ~ 1)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


z = np.array([-10, -2, 0, 2.3, 10])
print("z       :", z)
print("sigmoid :", sigmoid(z).round(3))


# 성질: z = 0 → 0.5, 부호를 바꾸면 1 - p
print("σ(0) =", sigmoid(0),
      "| σ(-2) + σ(2) =", round(sigmoid(-2) + sigmoid(2), 6))


# 가운데는 민감 · 양끝은 둔감: z를 1만큼 바꿀 때 확률 변화
for z0 in [0, 4]:
    print(f"z {z0} → {z0 + 1}: 확률 +{sigmoid(z0 + 1) - sigmoid(z0):.3f}")

z       : [-10.   -2.    0.    2.3  10. ]
sigmoid : [0.    0.119 0.5   0.909 1.   ]
σ(0) = 0.5 | σ(-2) + σ(2) = 1.0
z 0 → 1: 확률 +0.231
z 4 → 5: 확률 +0.011


**예측 정답** · 아님 — 가운데는 `+0.231`, 끝쪽은 `+0.011` (끝으로 갈수록 확신이 굳어 둔감)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>sigmoid : [0.    0.119 0.5   0.909 1.   ]</code></td>
<td style="text-align: left;">z = 0 → 0.5, z = 2.3 → 0.909 (원본 표의 예시)</td>
</tr>
<tr>
<td style="text-align: left;"><code>σ(0) = 0.5 &#124; σ(-2) + σ(2) = 1.0</code></td>
<td style="text-align: left;">부호를 바꾸면 반대 클래스 확률</td>
</tr>
<tr>
<td style="text-align: left;"><code>z 0 → 1: 확률 +0.231</code></td>
<td style="text-align: left;">경계 근처는 민감</td>
</tr>
<tr>
<td style="text-align: left;"><code>z 4 → 5: 확률 +0.011</code></td>
<td style="text-align: left;">양끝은 0 · 1에 붙어 둔감</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">점수 z (logit)</td>
<td style="text-align: left;">-∞ ~ +∞, 0이면 경계</td>
<td style="text-align: left;">모델 내부 계산 · 손실 계산 (6-1)</td>
</tr>
<tr>
<td style="text-align: left;">확률 σ(z)</td>
<td style="text-align: left;">0 ~ 1, 0.5이면 경계</td>
<td style="text-align: left;">사람에게 설명 · 기준 적용</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · `np.exp(-z)`는 z가 아주 작으면(예: -1000) 넘침 경고 — 실제 코드는 `scipy.special.expit` · `torch.sigmoid`처럼 안정화된 함수

> ✅ **한 줄 정리** · 로지스틱은 점수 z를 시그모이드 1 / (1 + e^-z)로 확률로 바꾸며, 기준 0.5는 점수 0과 같다.

**딥러닝 코드에서는** · 이진 분류 신경망의 마지막 층 = `nn.Linear(…, 1)` → 시그모이드 (6-1)

### 3-2. `A` 답과 확률 꺼내기 · `predict` · `predict_proba` <sub>원본 4-1 Part 3 셀 10 ~ 15 · 41 ~ 43</sub>

> 🎯 **`predict`와 `predict_proba`는 무엇이 다르며, 그 확률은 모델 안의 어떤 숫자에서 나오는가?**

💡 **쉽게 말하면** `predict`는 **합격 · 불합격 통지서**, `predict_proba`는 **합격 확률이 적힌 성적표**. 통지서는 성적표에 0.5 선을 그어 만든 것.

**실행 흐름** · valid 생존 확률 앞 5명 → 두 열의 합 → predict == 확률 ≥ 0.5 → decision_function(z) → σ(z)와 비교 → 반복 횟수

**먼저 예측** · `predict(X)`와 `predict_proba(X)[:, 1] >= 0.5`는 항상 같을까? — 답은 코드 실행 결과 아래에

In [8]:
# predict_proba → [사망 확률, 생존 확률], [:, 1] = Positive(생존) 확률
proba = lr_all.predict_proba(X_valid)[:, 1]
print("생존 확률 앞 5명:", proba[:5].round(4))
print("두 열의 합 = 1:", np.allclose(lr_all.predict_proba(X_valid).sum(1), 1))


# predict = 확률 ≥ 0.5 → 1
same = (lr_all.predict(X_valid) == (proba >= 0.5).astype(int)).all()
print("predict == (확률 ≥ 0.5):", same)


# 확률의 출처: 점수 z (decision_function) → 시그모이드
z_val = lr_all.decision_function(X_valid)
print("z 앞 5명:", z_val[:5].round(3))
print("σ(z) == predict_proba:", np.allclose(sigmoid(z_val), proba))
print("실제 반복 횟수:", lr_all.named_steps["model"].n_iter_[0],
      "/ max_iter 5000")

생존 확률 앞 5명: [0.1093 0.1176 0.74   0.1017 0.084 ]
두 열의 합 = 1: True
predict == (확률 ≥ 0.5): True
z 앞 5명: [-2.098 -2.015  1.046 -2.178 -2.389]
σ(z) == predict_proba: True
실제 반복 횟수: 32 / max_iter 5000


**예측 정답** · 같음 — `True` (predict는 기준 0.5를 자동으로 적용한 것)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>생존 확률 앞 5명: [0.1093 0.1176 0.74   0.1017 0.084 ]</code></td>
<td style="text-align: left;">원본 셀 12와 같은 값</td>
</tr>
<tr>
<td style="text-align: left;"><code>두 열의 합 = 1: True</code></td>
<td style="text-align: left;">[사망 확률, 생존 확률]</td>
</tr>
<tr>
<td style="text-align: left;"><code>predict == (확률 ≥ 0.5): True</code></td>
<td style="text-align: left;">predict = 기준 0.5짜리 판정</td>
</tr>
<tr>
<td style="text-align: left;"><code>z 앞 5명: [-2.098 -2.015  1.046 -2.178 -2.389]</code></td>
<td style="text-align: left;">경계선에서 떨어진 정도에 비례하는 점수</td>
</tr>
<tr>
<td style="text-align: left;"><code>σ(z) == predict_proba: True</code></td>
<td style="text-align: left;">확률의 출처는 점수 z</td>
</tr>
<tr>
<td style="text-align: left;"><code>실제 반복 횟수: 32 / max_iter 5000</code></td>
<td style="text-align: left;">max_iter는 상한일 뿐 (수렴 경고 방지용)</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>predict(X)</code></td>
<td style="text-align: left;">0 / 1 <strong>답만</strong> (안에서 확률 ≥ 0.5)</td>
<td style="text-align: left;">기준 0.5가 업무와 맞을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>predict_proba(X)[:, 1]</code></td>
<td style="text-align: left;">Positive <strong>확률</strong> → 기준을 직접 정할 수 있음</td>
<td style="text-align: left;">기준 조정 · AUC · 순위 매기기 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>decision_function(X)</code></td>
<td style="text-align: left;">시그모이드 전 <strong>점수 z</strong></td>
<td style="text-align: left;">점수 그대로 순위 · 손실 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · `predict_proba`의 열 순서는 `model.classes_` 순서 — 정답이 문자열이면 `[:, 1]`이 무엇인지 먼저 확인

> ✅ **한 줄 정리** · 기준을 직접 정하거나 순위를 매기려면 `predict` 대신 `predict_proba(X)[:, 1]`로 Positive 확률을 꺼낸다.

**딥러닝 코드에서는** · PyTorch 모델은 `predict`가 없음 → `logits = model(x)` → `prob = torch.sigmoid(logits)` → `pred = prob >= thr`를 직접 씀

### 3-3. `A` 확신하고 틀린 대가 · `log_loss` <sub>원본 4-1 셀 16 ~ 18 · 35 ~ 37</sub>

> 🎯 **정답이 0인데 0.51로 틀린 것과 0.99로 틀린 것은 벌점이 얼마나 다르며, 모델은 확신하면서도 틀리는가?**

💡 **쉽게 말하면** 교차 엔트로피는 **확신의 대가를 매기는 채점**. 살짝 틀리면 작은 벌점, 큰소리치고 틀리면 벌점 폭탄.

**실행 흐름** · -log(1 - q)로 q별 벌점 → 0.99 / 0.51 배수 → valid log loss → 확률 0.9 이상 · 0.1 이하인데 틀린 승객

**먼저 예측** · 0.99로 틀린 벌점은 0.51로 틀린 벌점의 몇 배일까? — 답은 코드 실행 결과 아래에

In [9]:
from sklearn.metrics import log_loss


# 원본 셀: 정답이 사망(0)인데 생존 확률 q로 예측 → 벌점 -log(1 - q)
for q in [0.51, 0.7, 0.9, 0.99]:
    print(f"생존 확률 {q:.2f}로 틀림 → 벌점 {-np.log(1 - q):.4f}")
print(f"0.99 벌점 / 0.51 벌점 = {np.log(1 - 0.99) / np.log(1 - 0.51):.2f}배")


# 모델 전체의 평균 벌점 = log loss (교차 엔트로피)
print(f"valid log loss: {log_loss(y_valid, proba):.4f}")


# 확신하고 틀린 승객 (확률 0.9 이상 · 0.1 이하인데 오답)
wrong = (proba >= 0.5).astype(int) != y_valid.to_numpy()
sure = (proba >= 0.9) | (proba <= 0.1)
print("틀린 예측:", wrong.sum(), "명 | 그중 확신하고 틀림:",
      (wrong & sure).sum(), "명")
print("그 확률들:", np.sort(proba[wrong & sure]).round(3))

생존 확률 0.51로 틀림 → 벌점 0.7133
생존 확률 0.70로 틀림 → 벌점 1.2040
생존 확률 0.90로 틀림 → 벌점 2.3026
생존 확률 0.99로 틀림 → 벌점 4.6052
0.99 벌점 / 0.51 벌점 = 6.46배
valid log loss: 0.4460
틀린 예측: 36 명 | 그중 확신하고 틀림: 5 명
그 확률들: [0.075 0.089 0.1   0.1   0.956]


**예측 정답** · `6.46`배 (0.7133 → 4.6052)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>생존 확률 0.51로 틀림 → 벌점 0.7133</code></td>
<td style="text-align: left;">거의 반반이면 작은 벌점</td>
</tr>
<tr>
<td style="text-align: left;"><code>생존 확률 0.99로 틀림 → 벌점 4.6052</code></td>
<td style="text-align: left;">확신하고 틀리면 급격히 커짐</td>
</tr>
<tr>
<td style="text-align: left;"><code>0.99 벌점 / 0.51 벌점 = 6.46배</code></td>
<td style="text-align: left;">원본 셀 17과 같은 값</td>
</tr>
<tr>
<td style="text-align: left;"><code>valid log loss: 0.4460</code></td>
<td style="text-align: left;">valid 승객 전체 벌점의 평균</td>
</tr>
<tr>
<td style="text-align: left;"><code>틀린 예측: 36 명 &#124; 그중 확신하고 틀림: 5 명</code></td>
<td style="text-align: left;">확률 0.9 이상 · 0.1 이하인데 오답</td>
</tr>
<tr>
<td style="text-align: left;"><code>그 확률들: [0.075 0.089 0.1   0.1   0.956]</code></td>
<td style="text-align: left;">확률 0.956인데 사망한 승객도 있음</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도</td>
<td style="text-align: left;">맞음 / 틀림만, 확신 정도는 무시</td>
<td style="text-align: left;">최종 판정 보고</td>
</tr>
<tr>
<td style="text-align: left;">log loss (교차 엔트로피)</td>
<td style="text-align: left;">확신하고 틀릴수록 벌점이 급격히 커짐</td>
<td style="text-align: left;">확률 품질 평가 · 모델 학습의 손실</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · 확률 0.97 ≠ '97% 맞음' — 모델이 그쪽으로 강하게 기울었다는 뜻이고, 실제 빈도와 맞는지는 보정(calibration)으로 따로 확인

> ✅ **한 줄 정리** · 로지스틱은 확신하고 틀릴수록 벌점이 커지는 교차 엔트로피로 학습하지만, 확률이 높다고 정답이 보장되지는 않는다.

**딥러닝 코드에서는** · 딥러닝 분류의 학습 손실이 바로 이 교차 엔트로피 — `loss.item()`이 0.446이면 이 표의 평균 벌점과 같은 뜻 (6-1)

### 3-4. `B` 계수로 근거 읽기 · `coef_` <sub>원본 4-1 셀 48 ~ 52</sub>

> 🎯 **로지스틱의 계수는 무엇을 알려 주며, 3등석 여성과 1등석 남성 중 누구의 생존 확률이 높은가?**

💡 **쉽게 말하면** 계수는 **특성마다 점수에 더하고 빼는 가산점표**. 부호가 방향, 확률 표가 실제 결과.

**실행 흐름** · Pclass + Sex만 쓴 로지스틱 → 계수 · 절편 → 등급 3 × 성별 2 = 6조합 확률

**먼저 예측** · 3등석 여성과 1등석 남성 중 생존 확률이 높은 쪽은? — 답은 코드 실행 결과 아래에

In [10]:
# 원본 셀: Pclass + Sex 두 열만 쓴 로지스틱 → 계수 꺼내기
lr_ps = make_lr(["Pclass", "Sex"]).fit(X_train, y_train)
names = lr_ps.named_steps["prep"].get_feature_names_out()
coefs = lr_ps.named_steps["model"].coef_[0]
print(dict(zip(names.tolist(), coefs.round(3).tolist())),
      "| 절편:", round(lr_ps.named_steps["model"].intercept_[0], 3))


# 6가지 조합의 생존 확률 (3등석 여성 vs 1등석 남성)
combo = pd.DataFrame({"Pclass": [1, 2, 3] * 2,
                      "Sex": ["female"] * 3 + ["male"] * 3})
combo["생존 확률"] = lr_ps.predict_proba(combo)[:, 1].round(3)
print(combo.to_string(index=False))

{'num__Pclass': -0.826, 'cat__Sex_female': 1.307, 'cat__Sex_male': -1.306} | 절편: -0.248
 Pclass    Sex  생존 확률
      1 female  0.913
      2 female  0.796
      3 female  0.592
      1   male  0.436
      2   male  0.223
      3   male  0.096


**예측 정답** · 3등석 여성 `0.592` > 1등석 남성 `0.436` — 등급을 두 단계 낮춰도 성별을 못 뒤집음

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>{&#x27;num__Pclass&#x27;: -0.826, &#x27;cat__Sex_female&#x27;: 1.307, &#x27;cat__Sex_male&#x27;: -1.306} &#124; 절편: -0.248</code></td>
<td style="text-align: left;">Pclass(등급 숫자)가 클수록 ↓, 여성 ↑ · 남성 ↓</td>
</tr>
<tr>
<td style="text-align: left;"><code>3 female  0.592</code></td>
<td style="text-align: left;">3등석 여성</td>
</tr>
<tr>
<td style="text-align: left;"><code>1   male  0.436</code></td>
<td style="text-align: left;">1등석 남성보다 높음</td>
</tr>
<tr>
<td style="text-align: left;"><code>3   male  0.096</code></td>
<td style="text-align: left;">가장 낮은 조합</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">계수 부호</td>
<td style="text-align: left;">+ → 확률을 올림, - → 내림</td>
<td style="text-align: left;">어느 방향으로 작용하는지 설명</td>
</tr>
<tr>
<td style="text-align: left;">계수 크기 비교</td>
<td style="text-align: left;">표준화 숫자 열과 0/1 원-핫 열은 단위가 달라 <strong>직접 비교 위험</strong></td>
<td style="text-align: left;">같은 단위끼리만, 아니면 확률 표로 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · Pclass는 표준화 숫자, Sex는 0/1 원-핫이라 계수 크기를 그대로 비교하면 안 됨 — 결론은 확률 표로

> ✅ **한 줄 정리** · 로지스틱 계수의 부호로 각 특성이 확률을 올리는지 내리는지 읽고, 크기 비교는 확률 표로 확인한다.

**딥러닝 코드에서는** · 딥러닝은 계수가 수백만 개라 이렇게 읽을 수 없음 → 모델 설명이 중요하면 로지스틱을 기준선으로 함께 둠

#### 3장 정리 · 소속별 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>1 / (1 + np.exp(-z))</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">시그모이드</td>
</tr>
<tr>
<td style="text-align: left;"><code>predict()</code> · <code>predict_proba()[:, 1]</code></td>
<td style="text-align: left;">분류 모델 메서드</td>
<td style="text-align: left;">0/1 · Positive 확률</td>
</tr>
<tr>
<td style="text-align: left;"><code>decision_function()</code></td>
<td style="text-align: left;">선형 모델 메서드</td>
<td style="text-align: left;">점수 z (logit)</td>
</tr>
<tr>
<td style="text-align: left;"><code>n_iter_</code> · <code>coef_</code> · <code>intercept_</code></td>
<td style="text-align: left;">LogisticRegression 속성</td>
<td style="text-align: left;">반복 횟수 · 계수 · 절편</td>
</tr>
<tr>
<td style="text-align: left;"><code>log_loss(y, proba)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">평균 교차 엔트로피</td>
</tr>
</tbody>
</table>

---
## 4. 임곗값

> **연결** · 3장에서 predict가 확률 ≥ 0.5라는 것을 확인했다 → 0.5는 정해진 답이 아니라 바꿀 수 있는 정책. 바꾸면 무엇이 변하는지 보고, 머신러닝-1의 규칙대로 validation에서 고름

**흐름 위치** · ⑥ 선택 → ⑦ 최종 보고 — 기준은 validation, test는 1회

### 4-1. `A` 기준을 바꾸면 · `proba >= 기준` <sub>원본 4-1 Part 4 셀 23 ~ 25 (04 Part 5 정의와 같음)</sub>

> 🎯 **같은 확률에 판정 기준(임곗값)만 0.3 · 0.5 · 0.7로 바꾸면 정밀도 · 재현율 · 실수 개수는 어떻게 움직이는가?**

💡 **쉽게 말하면** 임곗값은 **합격선**. 합격선을 낮추면 더 많이 붙여서 진짜 생존자를 덜 놓치지만, 엉뚱한 사람도 더 붙임.

**실행 흐름** · 로지스틱 valid 확률 → 기준 3개로 판정 → 정밀도 · 재현율 · F1 · FP · FN · 양성 예측 수 표

**먼저 예측** · 기준을 0.5 → 0.3으로 낮추면 놓침(FN)과 오탐(FP)은 각각? — 답은 코드 실행 결과 아래에

In [11]:
# 같은 확률(lr_all)에 기준만 바꿔 예측 (valid)
rows = []
for thr in [0.3, 0.5, 0.7]:
    y_hat = (proba >= thr).astype(int)  # 기준 이상 → 생존(1)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_hat).ravel()
    rows.append({"기준": thr, "정밀도": precision_score(y_valid, y_hat),
                 "재현율": recall_score(y_valid, y_hat),
                 "F1": f1_score(y_valid, y_hat), "FP(오탐)": fp,
                 "FN(놓침)": fn, "양성 예측": y_hat.sum()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

 기준   정밀도   재현율    F1  FP(오탐)  FN(놓침)  양성 예측
0.3 0.671 0.838 0.745      28      11     85
0.5 0.729 0.750 0.739      19      17     70
0.7 0.814 0.515 0.631       8      33     43


**예측 정답** · FN `17 → 11`로 줄고, FP `19 → 28`로 늘어남

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>0.3 0.671 0.838 0.745      28      11     85</code></td>
<td style="text-align: left;">많이 잡음 → 재현율 ↑ · 오탐 ↑</td>
</tr>
<tr>
<td style="text-align: left;"><code>0.5 0.729 0.750 0.739      19      17     70</code></td>
<td style="text-align: left;">기본 기준 (= predict)</td>
</tr>
<tr>
<td style="text-align: left;"><code>0.7 0.814 0.515 0.631       8      33     43</code></td>
<td style="text-align: left;">확실할 때만 → 정밀도 ↑ · 놓침 ↑</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">기준 낮춤 (0.3)</td>
<td style="text-align: left;">Positive를 많이 잡음 → 재현율 ↑ · 정밀도 ↓ (FP ↑)</td>
<td style="text-align: left;">놓치면 큰일인 업무</td>
</tr>
<tr>
<td style="text-align: left;">기준 0.5 (기본)</td>
<td style="text-align: left;"><code>predict</code>와 같음</td>
<td style="text-align: left;">특별한 업무 기준이 없을 때</td>
</tr>
<tr>
<td style="text-align: left;">기준 높임 (0.7)</td>
<td style="text-align: left;">확실할 때만 Positive → 정밀도 ↑ · 재현율 ↓ (FN ↑)</td>
<td style="text-align: left;">잘못 울리면 큰일인 업무</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · 기준을 바꾸면 **모델은 그대로**이고 판정만 바뀜 — 다시 학습할 필요 없음

> ✅ **한 줄 정리** · 기준을 낮추면 재현율이 오르고 정밀도가 내려가며, 기준은 모델이 아니라 어떤 실수를 감수할지 정하는 정책이다.

**딥러닝 코드에서는** · 딥러닝 이진 분류도 `prob >= 0.5`의 0.5를 업무에 맞게 바꿈 — 의료 영상은 낮춰서 놓침을 줄이는 경우가 많음

### 4-2. `A` 업무 기준으로 고르고 test 1회 · `validation 선택` <sub>원본 04 Part 5 셀 28 ~ 41 · 4-1 셀 26 ~ 29</sub>

> 🎯 **기준(임곗값)은 어느 데이터로, 어떤 순서로 골라야 하며, 기본 0.5와 업무 기준은 어떻게 다른가?**

💡 **쉽게 말하면** 합격선도 **모의고사(validation)로 정하고 수능(test)은 1번**. 그리고 '어떤 합격선이 좋은가'의 규칙은 성적표를 보기 **전에** 정함.

**실행 흐름** · digits 새 3분할 → PCA 10개 + 로지스틱 (train fit) → val 기준표 → 규칙 A (F1 최대) · 규칙 B (정밀도 ≥ 0.95) → 규칙 A 기준으로 test 1회

**먼저 예측** · F1 최대 규칙과 '정밀도 0.95 이상 중 재현율 최대' 규칙은 같은 기준을 고를까? — 답은 코드 실행 결과 아래에

In [12]:
from sklearn.decomposition import PCA


# 원본 Part 5: 기준을 고르는 실험이므로 새 3분할 (test는 마지막 1회)
X7_dev, X7_test, y7_dev, y7_test = train_test_split(
    X_dig, y7, test_size=0.2, stratify=y7, random_state=42)
X7_fit, X7_val, y7_fit, y7_val = train_test_split(
    X7_dev, y7_dev, test_size=0.25, stratify=y7_dev, random_state=42)


# 기준 차이가 보이도록 정보를 10개로 줄인 모델 (Pipeline째 train에만 fit)
pipe7 = Pipeline([("pca", PCA(n_components=10, random_state=42)),
                  ("lr", LogisticRegression(max_iter=5000, C=0.1))])
pipe7.fit(X7_fit, y7_fit)
p_val = pipe7.predict_proba(X7_val)[:, 1]


# 규칙을 먼저 선언 → validation 표에서 고르기
thrs = np.round(np.arange(0.1, 1.0, 0.1), 1)
table = pd.DataFrame([{
    "기준": t,
    "정밀도": precision_score(y7_val, p_val >= t, zero_division=0),
    "재현율": recall_score(y7_val, p_val >= t),
    "F1": f1_score(y7_val, p_val >= t)} for t in thrs]).round(3)
print(table.head(6).to_string(index=False))
t_f1 = table.loc[table["F1"].idxmax(), "기준"]  # 규칙 A: F1 최대
ok = table[table["정밀도"] >= 0.95]  # 규칙 B: 오탐 비용이 큰 업무
t_prec = ok.loc[ok["재현율"].idxmax(), "기준"]
print("규칙 A (F1 최대) →", t_f1, "| 규칙 B (정밀도 ≥ 0.95 중 재현율 최대) →",
      t_prec)


# 선언한 규칙 A의 기준 하나로 test를 처음이자 마지막으로 채점
p_test = pipe7.predict_proba(X7_test)[:, 1] >= t_f1
print(f"FINAL TEST (기준 {t_f1}): 정밀도"
      f" {precision_score(y7_test, p_test):.3f}"
      f" · 재현율 {recall_score(y7_test, p_test):.3f}"
      f" · F1 {f1_score(y7_test, p_test):.3f}")

 기준   정밀도   재현율    F1
0.1 0.854 0.972 0.909
0.2 0.897 0.972 0.933
0.3 0.921 0.972 0.946
0.4 0.919 0.944 0.932
0.5 0.943 0.917 0.930
0.6 0.939 0.861 0.899
규칙 A (F1 최대) → 0.3 | 규칙 B (정밀도 ≥ 0.95 중 재현율 최대) → 0.7
FINAL TEST (기준 0.3): 정밀도 0.919 · 재현율 0.944 · F1 0.932


**예측 정답** · 다름 — 규칙 A `0.3`, 규칙 B `0.7` (업무 규칙이 답을 바꿈)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>0.3 0.921 0.972 0.946</code></td>
<td style="text-align: left;">val F1 최대</td>
</tr>
<tr>
<td style="text-align: left;"><code>0.5 0.943 0.917 0.930</code></td>
<td style="text-align: left;">기본 기준 0.5 (F1은 더 낮음)</td>
</tr>
<tr>
<td style="text-align: left;"><code>규칙 A (F1 최대) → 0.3 &#124; 규칙 B (정밀도 ≥ 0.95 중 재현율 최대) → 0.7</code></td>
<td style="text-align: left;">같은 표에서 규칙에 따라 다른 기준</td>
</tr>
<tr>
<td style="text-align: left;"><code>FINAL TEST (기준 0.3): 정밀도 0.919 · 재현율 0.944 · F1 0.932</code></td>
<td style="text-align: left;">원본 셀 36과 같은 결과, test 결과를 보고 기준을 바꾸지 않음</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">기본 기준 0.5</td>
<td style="text-align: left;">모델이 정한 경계 그대로, 업무 비용과 무관</td>
<td style="text-align: left;">기준을 따로 정할 근거가 없을 때</td>
</tr>
<tr>
<td style="text-align: left;">업무 기준 · F1 최대 (규칙 A)</td>
<td style="text-align: left;">두 실수를 비슷하게 봄 → 0.3</td>
<td style="text-align: left;">FP · FN 비용이 비슷할 때</td>
</tr>
<tr>
<td style="text-align: left;">업무 기준 · 정밀도 ≥ 0.95 (규칙 B)</td>
<td style="text-align: left;">오탐을 줄이는 쪽 → 0.7</td>
<td style="text-align: left;">잘못 울리는 비용이 클 때</td>
</tr>
<tr>
<td style="text-align: left;">test로 기준 고르기</td>
<td style="text-align: left;">test가 검증이 되어 점수가 부풀려짐 (머신러닝-1 5-3)</td>
<td style="text-align: left;">쓰지 않음</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · 표를 본 뒤에 규칙을 정하면 규칙 자체가 데이터에 맞춰짐 — 규칙을 먼저 적고, 기준 선택도 Pipeline 밖 누수의 한 종류 (머신러닝-2 3-3)

> ✅ **한 줄 정리** · 기준은 먼저 선언한 업무 규칙으로 validation에서 고르고, 고른 기준 하나로 test를 마지막에 한 번만 채점한다.

**딥러닝 코드에서는** · 딥러닝은 `best.pt`를 val 손실로 고르고, 그 모델의 val 확률로 기준을 고른 뒤, test loader로 1회 — 같은 순서

### 4-3. `B` 줄다리기는 경향 · `양성 예측 수` <sub>원본 4-1 셀 29 ~ 31</sub>

> 🎯 **기준을 올리면 정밀도는 항상 오르는가?**

💡 **쉽게 말하면** 합격자가 몇 명 안 남으면 **한 명만 잘못 붙어도 비율이 크게 흔들림**. 그래서 '보통 오른다'이지 '반드시 오른다'가 아님.

**실행 흐름** · 기준 0.05 간격 19개 → 정밀도가 이전보다 내려간 곳 · 그때의 양성 예측 수

**먼저 예측** · 기준을 올렸는데 정밀도가 내려간 곳이 있을까? — 답은 코드 실행 결과 아래에

In [13]:
# 원본 셀: 0.05 간격으로 촘촘하게 → 기준을 올렸는데 정밀도가 내려가는 곳
fine = np.round(np.arange(0.05, 0.96, 0.05), 2)
prev, drops = None, []
for t in fine:
    y_hat = (proba >= t).astype(int)
    pr = precision_score(y_valid, y_hat, zero_division=0)
    if prev is not None and pr < prev:
        drops.append((float(t), round(prev, 4), round(pr, 4),
                      int(y_hat.sum())))
    prev = pr
print("정밀도가 내려간 곳 (기준, 이전, 이후, 양성 예측 수):")
for d in drops:
    print(" ", d)

정밀도가 내려간 곳 (기준, 이전, 이후, 양성 예측 수):
  (0.65, 0.8036, 0.7708, 48)
  (0.95, 0.9474, 0.9167, 12)


**예측 정답** · 있음 — `2`곳 (0.65 · 0.95)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>(0.65, 0.8036, 0.7708, 48)</code></td>
<td style="text-align: left;">48명 중 오답이 섞여 0.80 → 0.77</td>
</tr>
<tr>
<td style="text-align: left;"><code>(0.95, 0.9474, 0.9167, 12)</code></td>
<td style="text-align: left;">12명만 남아 1명 차이로 크게 흔들림</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">"기준을 올리면 정밀도가 오른다"</td>
<td style="text-align: left;"><strong>경향</strong> (대부분 성립)</td>
<td style="text-align: left;">설명할 때</td>
</tr>
<tr>
<td style="text-align: left;">양성 예측이 적은 구간</td>
<td style="text-align: left;">1명만 틀려도 비율이 크게 흔들림 → 역전 가능</td>
<td style="text-align: left;">기준이 높을수록 개수까지 함께 보기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · "기준을 올리면 정밀도는 반드시 오른다"는 틀린 말 — 표본이 작은 valid(178명)일수록 흔들림이 큼

> ✅ **한 줄 정리** · 기준을 올리면 정밀도가 보통 오르지만 양성 예측이 적은 구간에서는 내려갈 수도 있으므로 개수와 함께 본다.

**딥러닝 코드에서는** · 검증 세트가 작으면 기준을 0.01 단위로 세밀하게 고르는 것이 오히려 과적합 → 0.05 ~ 0.1 간격이면 충분

#### 4장 정리 · 소속별 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>(proba &gt;= thr).astype(int)</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">기준으로 판정</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.arange(0.1, 1.0, 0.1)</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">기준 후보</td>
</tr>
<tr>
<td style="text-align: left;"><code>table.loc[table["F1"].idxmax()]</code></td>
<td style="text-align: left;">pandas</td>
<td style="text-align: left;">규칙으로 기준 고르기</td>
</tr>
<tr>
<td style="text-align: left;"><code>PCA(n_components=)</code></td>
<td style="text-align: left;">sklearn.decomposition</td>
<td style="text-align: left;">특성 수 줄이기 (약한 모델 만들기용)</td>
</tr>
</tbody>
</table>

---
## 5. 곡선 지표

> **연결** · 4장에서 기준마다 성적이 달랐다 → 기준을 정하기 전에 모든 기준에서의 실력을 한 숫자로 요약하는 방법. 불균형에서는 두 곡선이 다르게 말함

**흐름 위치** · ⑥ 평가 — 기준과 무관한 순위 성능

### 5-1. `A` 기준 전체의 성적 · ROC-AUC · `roc_auc_score` <sub>원본 04 Part 6 셀 42 ~ 46 · 4-1 셀 44 AUC 정의</sub>

> 🎯 **ROC 곡선과 AUC는 무엇을 재며, AUC 0.857은 어떤 뜻인가?**

💡 **쉽게 말하면** AUC는 **줄 세우기 시험**. 생존자 한 명과 사망자 한 명을 무작위로 뽑았을 때, 모델이 생존자에게 더 높은 확률을 줄 확률.

**실행 흐름** · 타이타닉 ROC 곡선 점 → AUC → digits AUC · 전부 0점 모델 AUC → 생존 × 사망 모든 쌍을 직접 비교

**먼저 예측** · 모든 (생존자, 사망자) 쌍을 비교해 생존자 확률이 더 높은 비율을 세면 AUC와 같을까? — 답은 코드 실행 결과 아래에

In [14]:
from sklearn.metrics import roc_curve


# ROC 곡선: 기준을 1 → 0으로 내리며 (FPR, TPR=재현율) 점을 찍음
fpr, tpr, cut = roc_curve(y_valid, proba)
print("ROC 점 개수:", len(fpr), f"| 처음 ({fpr[0]:.0f}, {tpr[0]:.0f})",
      f"| 끝 ({fpr[-1]:.0f}, {tpr[-1]:.0f})")


# AUC = 곡선 아래 넓이
print(f"타이타닉 valid AUC: {roc_auc_score(y_valid, proba):.3f}")
print(f"digits valid AUC  : {roc_auc_score(y7_val, p_val):.3f}"
      f" | 전부 0점 모델: {roc_auc_score(y7_val, np.zeros(len(y7_val))):.3f}")


# AUC의 뜻: 생존자 한 명 · 사망자 한 명을 뽑으면 생존자 점수가 더 높을 확률
pos, neg = proba[y_valid == 1], proba[y_valid == 0]
pair = (pos[:, None] > neg[None, :]).mean()  # 모든 (생존, 사망) 쌍 비교
tie = (pos[:, None] == neg[None, :]).mean() / 2  # 동점은 절반
print(f"쌍 비교로 직접: {pair + tie:.3f}"
      f" ({len(pos)} × {len(neg)} = {len(pos) * len(neg)}쌍)")

ROC 점 개수: 76 | 처음 (0, 0) | 끝 (1, 1)
타이타닉 valid AUC: 0.857
digits valid AUC  : 0.997 | 전부 0점 모델: 0.500
쌍 비교로 직접: 0.857 (68 × 110 = 7480쌍)


**예측 정답** · 같음 — 둘 다 `0.857`

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>ROC 점 개수: 76 &#124; 처음 (0, 0) &#124; 끝 (1, 1)</code></td>
<td style="text-align: left;">기준 1(아무도 Positive 아님) → 기준 0(전부 Positive)</td>
</tr>
<tr>
<td style="text-align: left;"><code>타이타닉 valid AUC: 0.857</code></td>
<td style="text-align: left;">1-1에서 정확도로는 규칙과 같았던 모델 → 규칙은 확률이 없어 AUC를 잴 수 없음</td>
</tr>
<tr>
<td style="text-align: left;"><code>digits valid AUC  : 0.997 &#124; 전부 0점 모델: 0.500</code></td>
<td style="text-align: left;">0.5 = 찍기 (순위 정보 없음)</td>
</tr>
<tr>
<td style="text-align: left;"><code>쌍 비교로 직접: 0.857 (68 × 110 = 7480쌍)</code></td>
<td style="text-align: left;">AUC의 뜻 그대로 계산</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도 · F1 (기준 하나)</td>
<td style="text-align: left;"><strong>한 기준</strong>에서의 성적</td>
<td style="text-align: left;">기준을 정한 뒤 최종 보고</td>
</tr>
<tr>
<td style="text-align: left;">ROC-AUC (기준 전체)</td>
<td style="text-align: left;">모든 기준에서 Positive를 Negative보다 <strong>위에 세우는 능력</strong>, 0.5 = 찍기</td>
<td style="text-align: left;">기준을 정하기 전 모델 비교 (클래스가 비교적 고를 때)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · AUC는 기준과 무관한 **순위** 성능 — AUC가 높아도 실제 판정 성적은 고른 기준에서 따로 확인

> ✅ **한 줄 정리** · ROC-AUC는 모든 기준에서 Positive를 Negative보다 위에 세우는 능력이며, 기준을 정하기 전에 모델을 비교할 때 쓴다.

**딥러닝 코드에서는** · 이미지 이상 탐지 · 의료 분류 논문의 주 지표 — `roc_auc_score(y, torch.sigmoid(logits).numpy())`

### 5-2. `A` 드문 Positive에서는 PR-AUC · `average_precision_score` <sub>원본 04 셀 47 ~ 51</sub>

> 🎯 **ROC-AUC와 PR-AUC(AP)는 무엇이 다르며, Positive가 드물 때 어느 쪽을 믿어야 하는가?**

💡 **쉽게 말하면** ROC는 **음성이 많으면 오탐이 좀 늘어도 비율이 거의 안 변하는 저울**, PR은 **'Positive라고 한 것 중 진짜'를 직접 다는 저울**. 양성이 귀할수록 PR이 정직함.

**실행 흐름** · PCA 10개(강한) · 3개(약한) 모델의 ROC-AUC · AP → 약한 모델에서 양성만 36 · 10 · 4장으로 줄여 다시

**먼저 예측** · 약한 모델의 ROC-AUC가 0.961이면 AP도 0.9 근처일까? — 답은 코드 실행 결과 아래에

In [15]:
# 일부러 약한 모델: 픽셀 정보를 3개로만 줄임
weak = Pipeline([("pca", PCA(n_components=3, random_state=42)),
                 ("lr", LogisticRegression(max_iter=5000, C=0.1))])
p_weak = weak.fit(X7_fit, y7_fit).predict_proba(X7_val)[:, 1]
for name, p in [("PCA 10개", p_val), ("PCA 3개", p_weak)]:
    print(f"{name}: ROC-AUC {roc_auc_score(y7_val, p):.3f}"
          f" · PR-AP {average_precision_score(y7_val, p):.3f}")


# 같은 약한 모델 · 같은 점수에서 7(양성)만 덜어내 더 드물게 만들면
rng = np.random.default_rng(0)
pos_idx, neg_idx = np.where(y7_val == 1)[0], np.where(y7_val == 0)[0]
for n_pos in [36, 10, 4]:
    keep = np.concatenate([neg_idx, rng.choice(pos_idx, n_pos, replace=False)])
    yk, pk = y7_val[keep], p_weak[keep]
    print(f"양성 {n_pos:>2}장 ({yk.mean():.1%}): ROC-AUC"
          f" {roc_auc_score(yk, pk):.3f} · PR-AP"
          f" {average_precision_score(yk, pk):.3f}")

PCA 10개: ROC-AUC 0.997 · PR-AP 0.979
PCA 3개: ROC-AUC 0.961 · PR-AP 0.701
양성 36장 (10.0%): ROC-AUC 0.961 · PR-AP 0.701
양성 10장 (3.0%): ROC-AUC 0.969 · PR-AP 0.481
양성  4장 (1.2%): ROC-AUC 0.954 · PR-AP 0.349


**예측 정답** · 아님 — AP `0.701`, 양성을 4장(1.2%)으로 줄이면 ROC-AUC는 `0.954`로 그대로인데 AP는 `0.349`

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>PCA 10개: ROC-AUC 0.997 · PR-AP 0.979</code></td>
<td style="text-align: left;">강한 모델은 둘 다 높음</td>
</tr>
<tr>
<td style="text-align: left;"><code>PCA 3개: ROC-AUC 0.961 · PR-AP 0.701</code></td>
<td style="text-align: left;">ROC만 보면 훌륭해 보이는 약한 모델</td>
</tr>
<tr>
<td style="text-align: left;"><code>양성 10장 (3.0%): ROC-AUC 0.969 · PR-AP 0.481</code></td>
<td style="text-align: left;">양성이 드물어지자 AP만 떨어짐</td>
</tr>
<tr>
<td style="text-align: left;"><code>양성  4장 (1.2%): ROC-AUC 0.954 · PR-AP 0.349</code></td>
<td style="text-align: left;">ROC는 거의 그대로 → 불균형에서 착시</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">ROC-AUC</td>
<td style="text-align: left;">FPR = FP / 전체 음성 → 음성이 많으면 FP가 늘어도 <strong>둔감</strong></td>
<td style="text-align: left;">균형 데이터 · 전체 순위 능력</td>
</tr>
<tr>
<td style="text-align: left;">PR-AUC (AP)</td>
<td style="text-align: left;">정밀도를 직접 봄 → Positive가 드물수록 <strong>약점이 드러남</strong>, 기준선 = 양성 비율</td>
<td style="text-align: left;">불균형 데이터 (양성 10% 이하) · 상위 순위 품질</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · AP의 찍기 기준선은 0.5가 아니라 **양성 비율**(10%면 0.1) — AP 0.35가 1.2% 데이터에서는 기준선의 약 29배

> ✅ **한 줄 정리** · Positive가 드문 불균형 데이터에서는 ROC-AUC가 높게 유지되므로, PR-AUC(AP)로 양성 쪽 성능을 확인한다.

**딥러닝 코드에서는** · 객체 탐지의 mAP가 바로 이 AP의 평균 — 클래스마다 PR 곡선 아래 넓이를 구해 평균

#### 5장 정리 · 소속별 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>roc_curve(y, proba)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">기준별 (FPR, TPR) 점</td>
</tr>
<tr>
<td style="text-align: left;"><code>roc_auc_score(y, proba)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">ROC 곡선 아래 넓이</td>
</tr>
<tr>
<td style="text-align: left;"><code>average_precision_score(y, proba)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">PR 곡선 요약 (AP)</td>
</tr>
<tr>
<td style="text-align: left;"><code>pos[:, None] &gt; neg[None, :]</code></td>
<td style="text-align: left;">numpy broadcasting</td>
<td style="text-align: left;">모든 쌍 비교</td>
</tr>
</tbody>
</table>

---
## 6. 딥러닝 연결

> **연결** · 1 ~ 5장은 sklearn이었다 → 로지스틱의 점수 · 시그모이드 · 교차 엔트로피가 신경망 이진 분류의 출력층 · 손실과 같고, 불균형 대처도 같은 원리

**흐름 위치** · ④ 모델 출력층 · ⑤ 손실 — logit · BCE · pos_weight

### 6-1. `A` 시그모이드 = 이진 분류 출력층 · `BCEWithLogitsLoss` <sub>추가 · 원본 4-1 손실 설명 연결</sub>

> 🎯 **로지스틱 회귀는 신경망의 무엇과 같으며, PyTorch의 이진 분류 손실은 sklearn의 무엇과 같은가?**

💡 **쉽게 말하면** 로지스틱 = **층이 하나뿐인 신경망**. 입력에 가중치를 곱해 더한 점수(logit)를 시그모이드에 통과시키고, 교차 엔트로피로 채점.

**실행 흐름** · 3-2의 점수 z를 텐서로 → torch.sigmoid와 predict_proba 비교 → BCEWithLogits · BCE · log_loss 비교 → 확률 ≥ 0.5 vs logit ≥ 0

**먼저 예측** · PyTorch `BCEWithLogitsLoss`와 sklearn `log_loss`는 같은 값일까? — 답은 코드 실행 결과 아래에

In [16]:
import torch  # 이진 분류 출력층과 손실 확인용


# 로지스틱 = 입력 → Linear(점수 z = logit) → sigmoid → 확률
logits = torch.tensor(z_val, dtype=torch.float32)  # 3-2의 z
target = torch.tensor(y_valid.to_numpy(), dtype=torch.float32)
p_torch = torch.sigmoid(logits)
print("torch.sigmoid == predict_proba:",
      np.allclose(p_torch.numpy(), proba, atol=1e-6))


# 손실: BCEWithLogitsLoss(점수) = BCELoss(확률) = sklearn log_loss
bce_logit = torch.nn.BCEWithLogitsLoss()(logits, target)
bce_prob = torch.nn.BCELoss()(p_torch, target)
print(f"BCEWithLogits {bce_logit.item():.4f} · BCE {bce_prob.item():.4f}"
      f" · log_loss {log_loss(y_valid, proba):.4f}")


# 기준 0.5 = 점수 0 → 확률을 만들지 않고 logit만으로도 같은 예측
print("(확률 ≥ 0.5) == (logit ≥ 0):",
      torch.equal(p_torch >= 0.5, logits >= 0))

torch.sigmoid == predict_proba: True
BCEWithLogits 0.4460 · BCE 0.4460 · log_loss 0.4460
(확률 ≥ 0.5) == (logit ≥ 0): True


**예측 정답** · 같음 — 셋 다 `0.4460`

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>torch.sigmoid == predict_proba: True</code></td>
<td style="text-align: left;">같은 점수 → 같은 확률</td>
</tr>
<tr>
<td style="text-align: left;"><code>BCEWithLogits 0.4460 · BCE 0.4460 · log_loss 0.4460</code></td>
<td style="text-align: left;">3-3의 평균 벌점과 같은 값</td>
</tr>
<tr>
<td style="text-align: left;"><code>(확률 ≥ 0.5) == (logit ≥ 0): True</code></td>
<td style="text-align: left;">기준 0.5 = 점수 0</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>Linear</code> + <code>BCEWithLogitsLoss</code></td>
<td style="text-align: left;">점수(logit)를 바로 넣음, 안에서 시그모이드 + 안정적 계산</td>
<td style="text-align: left;">이진 분류 학습 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>sigmoid</code> + <code>BCELoss</code></td>
<td style="text-align: left;">확률을 넣음, 0 · 1에 가까우면 계산이 불안정</td>
<td style="text-align: left;">쓰지 않음 (출력을 확률로 보여 줄 때만 <code>sigmoid</code>)</td>
</tr>
<tr>
<td style="text-align: left;">다중 분류</td>
<td style="text-align: left;"><code>softmax</code> + <code>CrossEntropyLoss</code> (클래스 3개 이상)</td>
<td style="text-align: left;">이미지 10종 분류 등</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · `BCEWithLogitsLoss`에는 시그모이드를 거치지 않은 **logit**을 넣음 — 모델 끝에 sigmoid를 붙이고 이 손실을 쓰면 시그모이드가 두 번 적용됨

> ✅ **한 줄 정리** · 이진 분류 신경망은 Linear → logit → 시그모이드 확률 구조이며, `BCEWithLogitsLoss`는 sklearn의 log loss와 같은 손실이다.

**딥러닝 코드에서는** · `model = nn.Sequential(..., nn.Linear(64, 1))` · `loss_fn = nn.BCEWithLogitsLoss()` · 예측은 `torch.sigmoid(model(x)) >= thr`

### 6-2. `A` 불균형에서 지표와 가중치 · `class_weight` · `pos_weight` <sub>추가 · 원본 04 Part 2 불균형 연결</sub>

> 🎯 **Positive가 드물 때 학습 단계에서는 무엇을 조정하며, 그 효과는 어떤 지표로 확인하는가?**

💡 **쉽게 말하면** 소수 클래스의 실수에 **벌점을 더 크게 매겨** 모델이 그쪽을 신경 쓰게 만드는 것. 효과는 정확도가 아니라 재현율 · 정밀도로 확인.

**실행 흐름** · digits train 음성 · 양성 수 → pos_weight → 약한 모델을 class_weight 없음 · balanced로 학습 → 기준 0.5에서 3개 지표 → BCE pos_weight 벌점

**먼저 예측** · `class_weight="balanced"`로 학습하면 정확도는 오를까? — 답은 코드 실행 결과 아래에

In [17]:
# 불균형 digits(7 = 10%)에서 양성 가중치: 음성 수 / 양성 수
n_neg, n_pos = int((y7_fit == 0).sum()), int((y7_fit == 1).sum())
pos_w = n_neg / n_pos
print(f"train 음성 {n_neg} · 양성 {n_pos} → pos_weight {pos_w:.2f}")


# 같은 약한 모델: 가중치 없음 vs class_weight="balanced" (기준 0.5 그대로)
for cw in [None, "balanced"]:
    m = Pipeline([("pca", PCA(n_components=3, random_state=42)),
                  ("lr", LogisticRegression(max_iter=5000, C=0.1,
                                            class_weight=cw))])
    p = m.fit(X7_fit, y7_fit).predict(X7_val)
    print(f"class_weight={str(cw):<8} 정확도 {accuracy_score(y7_val, p):.3f}"
          f" · 정밀도 {precision_score(y7_val, p, zero_division=0):.3f}"
          f" · 재현율 {recall_score(y7_val, p):.3f}")


# 딥러닝에서 같은 일: 양성 손실에 가중치
loss_w = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w]))
one = (torch.tensor([-1.0]), torch.tensor([1.0]))  # 양성인데 점수 -1
print(f"양성 1개를 놓친 벌점: 기본 {torch.nn.BCEWithLogitsLoss()(*one):.3f}"
      f" → pos_weight {loss_w(*one):.3f}")

train 음성 970 · 양성 107 → pos_weight 9.07


class_weight=None     정확도 0.933 · 정밀도 0.731 · 재현율 0.528


class_weight=balanced 정확도 0.917 · 정밀도 0.556 · 재현율 0.833
양성 1개를 놓친 벌점: 기본 1.313 → pos_weight 11.905


**예측 정답** · 아님 — 정확도 `0.933 → 0.917`로 내려가지만 재현율 `0.528 → 0.833`으로 크게 오름

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>train 음성 970 · 양성 107 → pos_weight 9.07</code></td>
<td style="text-align: left;">음성이 양성의 9배</td>
</tr>
<tr>
<td style="text-align: left;"><code>class_weight=None     정확도 0.933 · 정밀도 0.731 · 재현율 0.528</code></td>
<td style="text-align: left;">7의 절반 가까이 놓침</td>
</tr>
<tr>
<td style="text-align: left;"><code>class_weight=balanced 정확도 0.917 · 정밀도 0.556 · 재현율 0.833</code></td>
<td style="text-align: left;">놓침은 줄고 오탐은 늘어남 (4-1의 기준 낮추기와 같은 방향)</td>
</tr>
<tr>
<td style="text-align: left;"><code>양성 1개를 놓친 벌점: 기본 1.313 → pos_weight 11.905</code></td>
<td style="text-align: left;">양성 실수의 벌점이 약 9배</td>
</tr>
</tbody>
</table>

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">가중치 없음</td>
<td style="text-align: left;">다수 클래스 쪽으로 기울어 재현율이 낮음</td>
<td style="text-align: left;">균형 데이터</td>
</tr>
<tr>
<td style="text-align: left;"><code>class_weight="balanced"</code> · <code>pos_weight</code></td>
<td style="text-align: left;">소수 클래스의 실수에 큰 벌점 → 학습 단계에서 재현율 ↑</td>
<td style="text-align: left;">불균형 + 놓침 비용이 클 때</td>
</tr>
<tr>
<td style="text-align: left;">기준(임곗값) 낮추기</td>
<td style="text-align: left;">학습은 그대로, 판정 단계에서 재현율 ↑</td>
<td style="text-align: left;">다시 학습하기 어려울 때 · 가중치와 함께</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · 가중치를 주면 확률 자체가 양성 쪽으로 밀려 확률 해석이 달라짐 — 기준은 가중치를 준 모델의 validation에서 다시 고르기

> ✅ **한 줄 정리** · 불균형 데이터는 `class_weight` · `pos_weight`로 소수 클래스의 실수에 큰 벌점을 주고, 효과는 정확도가 아니라 재현율 · 정밀도 · PR-AUC로 확인한다.

**딥러닝 코드에서는** · `nn.BCEWithLogitsLoss(pos_weight=torch.tensor([음성/양성]))`, 다중 분류는 `nn.CrossEntropyLoss(weight=클래스별 가중치)`

#### 6장 정리 · 소속별 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>torch.sigmoid()</code></td>
<td style="text-align: left;">torch</td>
<td style="text-align: left;">logit → 확률</td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.BCEWithLogitsLoss(pos_weight=)</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">logit 입력 이진 교차 엔트로피 · 양성 가중치</td>
</tr>
<tr>
<td style="text-align: left;"><code>nn.BCELoss()</code></td>
<td style="text-align: left;">torch.nn</td>
<td style="text-align: left;">확률 입력 (잘 안 씀)</td>
</tr>
<tr>
<td style="text-align: left;"><code>LogisticRegression(class_weight="balanced")</code></td>
<td style="text-align: left;">sklearn.linear_model</td>
<td style="text-align: left;">클래스 비율 반대로 가중치</td>
</tr>
</tbody>
</table>

---
## 7. 종합 실습 · 타이타닉 전체 흐름

> **연결** · 머신러닝-1(분할 · 교차검증 · test 1회) + 머신러닝-2(파이프라인) + 머신러닝-3(지표 · 임곗값)을 한 번에. 원본 4-1 Part 6 '수능 보러 가기'와 같은 자리이며, 머신러닝-2 6장에서 봉인한 test 179명을 여기서 처음 엶

**흐름 위치** · ② 분할 → ③ 파이프라인 → ⑤ 교차검증 → ⑥ 임곗값 선택 → ⑦ 최종 test 1회

> 🎯 **분할부터 최종 test까지, 무엇을 어느 데이터로 고르고 test는 언제 한 번 여는가?**

💡 **쉽게 말하면** 교과서(train)로 공부하며 모의고사(교차검증)로 **공부법**(모델)을 고르고, 별도 모의고사지(valid)로 **합격선**(기준)을 정한 뒤, 봉인한 수능(test)을 딱 한 번 봄. 각 결정에 쓰는 데이터가 겹치지 않는 것이 핵심.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 데이터</th>
<th style="text-align: left;">앞 권</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">분할 확인 · 생존 비율이 세 덩어리에서 비슷한가</td>
<td style="text-align: left;">전체</td>
<td style="text-align: left;">머신러닝-1 2-1</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">파이프라인 후보 3개 (로지스틱 C 2개 · 트리)</td>
<td style="text-align: left;">—</td>
<td style="text-align: left;">머신러닝-2 6장</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">5-Fold로 AUC · F1 · 정확도 → <strong>AUC로 고르기로 먼저 선언</strong></td>
<td style="text-align: left;">train</td>
<td style="text-align: left;">머신러닝-1 3-4</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">고른 모델의 valid 확률로 기준 고르기 (규칙: F1 최대, 0.05 간격)</td>
<td style="text-align: left;">valid</td>
<td style="text-align: left;">4-2</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">모델 · 전처리 · 기준을 고정하고 test 1회</td>
<td style="text-align: left;">test</td>
<td style="text-align: left;">머신러닝-1 5-1</td>
</tr>
</tbody>
</table>

**먼저 예측** · valid에서 고른 기준의 F1은 test에서도 그대로 나올까? — 답은 코드 실행 결과 아래에

In [18]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier


# 1단계: 분할 확인 (0장의 3분할 · test는 아직 한 번도 안 씀)
print(f"train {len(X_train)} · valid {len(X_valid)} · test {len(X_test)}"
      f" · 생존 비율 {y_train.mean():.3f} / {y_valid.mean():.3f}"
      f" / {y_test.mean():.3f}")


# 2단계: 파이프라인 후보 (트리는 스케일링 없이, 머신러닝-2 6장)
tree_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
     CAT)])
cands = {"LR C=0.1": make_lr(C=0.1), "LR C=1": make_lr(C=1.0),
         "Tree 깊이 4": Pipeline([
             ("prep", tree_prep),
             ("model", DecisionTreeClassifier(max_depth=4,
                                              random_state=42))])}


# 3단계: train 안에서 5-Fold · 지표 3개 (AUC로 고르기로 먼저 선언)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for name, pipe in cands.items():
    r = cross_validate(pipe, X_train, y_train, cv=cv5,
                       scoring=["roc_auc", "f1", "accuracy"])
    print(f"{name:<10} AUC {r['test_roc_auc'].mean():.3f}"
          f" · F1 {r['test_f1'].mean():.3f}"
          f" · 정확도 {r['test_accuracy'].mean():.3f}")

train 534 · valid 178 · test 179 · 생존 비율 0.384 / 0.382 / 0.385


LR C=0.1   AUC 0.853 · F1 0.749 · 정확도 0.815


LR C=1     AUC 0.851 · F1 0.747 · 정확도 0.805
Tree 깊이 4  AUC 0.820 · F1 0.706 · 정확도 0.792


In [19]:
# 4단계: AUC 1위 LR C=0.1 → train으로 fit → valid 확률로 기준 고르기 (F1 최대)
final = make_lr(C=0.1).fit(X_train, y_train)
p_v = final.predict_proba(X_valid)[:, 1]
grid_t = np.round(np.arange(0.1, 0.91, 0.05), 2)
f1s = [f1_score(y_valid, p_v >= t) for t in grid_t]
best_t = float(grid_t[int(np.argmax(f1s))])
print(f"valid: AUC {roc_auc_score(y_valid, p_v):.3f}"
      f" · F1(0.5) {f1_score(y_valid, p_v >= 0.5):.3f}"
      f" → 고른 기준 {best_t} · F1 {max(f1s):.3f}")


# 5단계: 모델 · 전처리 · 기준을 모두 고정한 뒤 test 1회
p_t = final.predict_proba(X_test)[:, 1]
y_hat = p_t >= best_t
print(f"TEST (기준 {best_t}): 정확도 {accuracy_score(y_test, y_hat):.3f}"
      f" · 정밀도 {precision_score(y_test, y_hat):.3f}"
      f" · 재현율 {recall_score(y_test, y_hat):.3f}"
      f" · F1 {f1_score(y_test, y_hat):.3f}")
print(f"TEST AUC: {roc_auc_score(y_test, p_t):.3f} (기준과 무관한 순위 성능)")

valid: AUC 0.856 · F1(0.5) 0.718 → 고른 기준 0.35 · F1 0.750
TEST (기준 0.35): 정확도 0.754 · 정밀도 0.651 · 재현율 0.783 · F1 0.711
TEST AUC: 0.835 (기준과 무관한 순위 성능)


**예측 정답** · 아님 — valid F1 `0.750` → test F1 `0.711` (고를 때 쓴 점수는 낙관적, 머신러닝-1 5-2와 같은 현상)

**결과 해설**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">출력</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>train 534 · valid 178 · test 179 · 생존 비율 0.384 / 0.382 / 0.385</code></td>
<td style="text-align: left;">stratify 덕분에 세 덩어리의 생존 비율이 같음</td>
</tr>
<tr>
<td style="text-align: left;"><code>LR C=0.1   AUC 0.853 · F1 0.749 · 정확도 0.815</code></td>
<td style="text-align: left;">AUC 1위 → 선언대로 선택</td>
</tr>
<tr>
<td style="text-align: left;"><code>LR C=1     AUC 0.851 · F1 0.747 · 정확도 0.805</code></td>
<td style="text-align: left;">차이 0.002 → 사실상 같음 (머신러닝-1 3-5)</td>
</tr>
<tr>
<td style="text-align: left;"><code>Tree 깊이 4  AUC 0.820 · F1 0.706 · 정확도 0.792</code></td>
<td style="text-align: left;">세 지표 모두 가장 낮음</td>
</tr>
<tr>
<td style="text-align: left;"><code>valid: AUC 0.856 · F1(0.5) 0.718 → 고른 기준 0.35 · F1 0.750</code></td>
<td style="text-align: left;">기준을 0.35로 낮추면 valid F1이 0.032 오름</td>
</tr>
<tr>
<td style="text-align: left;"><code>TEST (기준 0.35): 정확도 0.754 · 정밀도 0.651 · 재현율 0.783 · F1 0.711</code></td>
<td style="text-align: left;">보고할 최종 성적 (재현율을 높인 기준)</td>
</tr>
<tr>
<td style="text-align: left;"><code>TEST AUC: 0.835 (기준과 무관한 순위 성능)</code></td>
<td style="text-align: left;">CV 0.853 · valid 0.856보다 약간 낮음 (원본 4-1의 test AUC 0.829와 비슷)</td>
</tr>
</tbody>
</table>

**해석** · test 점수가 valid보다 낮게 나왔지만 이것이 정직한 성적이며, **이 결과를 보고 기준을 0.5로 되돌리거나 모델을 바꾸지 않음** — 그 순간 test가 검증이 되어 머신러닝-1 5-3의 부풀림이 생김. 개선하려면 새 데이터 또는 train 안의 교차검증으로 다시 고르고, 그때는 새 test가 필요함. 기준 0.35는 '생존자를 덜 놓치는' 쪽(재현율 0.783)이라 업무가 오탐을 싫어한다면 4-2의 규칙 B처럼 다른 규칙을 **처음부터** 선언했어야 함.

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">교차검증 (train 안)</td>
<td style="text-align: left;">모델 · 전처리 · 하이퍼파라미터 고르기</td>
<td style="text-align: left;">후보 비교 (여러 번)</td>
</tr>
<tr>
<td style="text-align: left;">valid (따로 떼어 둔 덩어리)</td>
<td style="text-align: left;">기준(임곗값) 고르기 · 고른 모델 확인</td>
<td style="text-align: left;">선택 (여러 번 봐도 됨)</td>
</tr>
<tr>
<td style="text-align: left;">test</td>
<td style="text-align: left;">모든 선택을 고정한 뒤 보고용 성적</td>
<td style="text-align: left;">마지막 1번 (결과로 다시 고르지 않음)</td>
</tr>
<tr>
<td style="text-align: left;">CV로 기준까지 고르기</td>
<td style="text-align: left;">train 안의 OOF 확률(<code>cross_val_predict</code>)로 기준 선택 → valid 덩어리를 아낄 수 있음</td>
<td style="text-align: left;">데이터가 적을 때의 대안</td>
</tr>
</tbody>
</table>

> ⚠️ **주의** · "test 결과를 보니 0.5가 더 좋다"는 비교 자체를 하지 않기 — 기준을 바꿀 근거는 validation에서만 나옴

> ✅ **한 줄 정리** · 모델은 train의 교차검증으로, 기준은 valid로, 각각 먼저 선언한 규칙에 따라 고르고, 모든 선택을 고정한 뒤 test를 한 번만 열어 보고한다.

**딥러닝 코드에서는** · 같은 5단계 — `random_split`으로 3분할 → `Dataset`의 transform = 파이프라인 → epoch마다 val 손실 · AUC로 `best.pt` 저장 → `best.pt`의 val 확률로 기준 선택 → test loader로 1회 평가 후 보고

---
## 8. 백지 복습

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 8-1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">정확도가 같은 두 모델은 같은 실력인가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">양성이 10%인 데이터에서 전부 음성으로 찍으면 정확도 · 재현율은?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">혼동행렬의 4칸과 <code>ravel()</code> 순서는?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">정밀도 · 재현율은 각각 어떤 실수를 보며, 공식은?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">macro와 weighted 평균은 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">시그모이드에서 확률 0.5는 점수 z로 얼마인가?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;"><code>predict</code>와 <code>predict_proba</code>의 관계는?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">확률 0.97은 '97% 맞다'는 뜻인가?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">기준을 낮추면 정밀도 · 재현율 · FP · FN은 어떻게 변하는가?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">기준은 어느 데이터로, 어떤 순서로 고르는가?</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">기준을 올리면 정밀도는 항상 오르는가?</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">AUC 0.857의 뜻은?</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">불균형 데이터에서 ROC-AUC와 PR-AUC 중 무엇을 보는가?</td>
</tr>
<tr>
<td style="text-align: left;">6-1</td>
<td style="text-align: left;"><code>BCEWithLogitsLoss</code>에는 무엇을 넣는가?</td>
</tr>
<tr>
<td style="text-align: left;">6-2</td>
<td style="text-align: left;">불균형 데이터에서 학습 단계에 조정할 것은?</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">모델 · 기준 · 최종 성적은 각각 어느 데이터로 정하는가?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**7** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 8-2. 코드로 답하기

원본 로지스틱 노트북의 실험(특성 조합 AUC · 성별 × 아이)과 지표 공식을 문제로 바꾼 것. 아래 빈 셀에 직접 작성 후 실행.

1. TN 50 · FP 10 · FN 5 · TP 35일 때 정확도 · 정밀도 · 재현율 · F1을 **공식으로** 계산 (2-2)
2. `y = [0, 1, 0, 1, 1, 0, 1, 0]`, 확률 `p = [0.2, 0.45, 0.6, 0.8, 0.35, 0.1, 0.9, 0.4]`에서 기준 **0.5와 0.3**의 정밀도 · 재현율 (4-1)
3. 2번 데이터로 '양성 점수 > 음성 점수'인 쌍의 비율을 직접 세고 `roc_auc_score`와 비교 (5-1)
4. 타이타닉(나이 결측 제외)에서 **성별 × 아이(12세 이하)** 생존률 4개 (원본 4-1 셀 46)
5. 타이타닉 3분할(0장과 같은 seed)에서 **Sex 하나만 · Pclass 하나만** 쓴 로지스틱의 valid AUC (원본 4-1 셀 51)

In [20]:
# 8-2 직접 작성

### 8-3. 모범 답안 · 8-1 · 8-2를 다 쓴 뒤에 보기

#### 8-1 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">아님. 정확도는 맞힌 개수만 셈 (규칙 한 줄 0.798 = 로지스틱 0.798, 예측이 다른 승객 22명)</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">정확도 0.900, 재현율 0 (양성을 하나도 못 찾음)</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">행 = 실제, 열 = 예측 → <code>tn, fp, fn, tp</code></td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">정밀도 TP / (TP + FP) = 오탐, 재현율 TP / (TP + FN) = 놓침</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">macro는 클래스마다 같은 비중, weighted는 샘플 수 비중</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">z = 0</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;"><code>predict</code> = <code>predict_proba[:, 1] &gt;= 0.5</code></td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">아님. 그쪽으로 강하게 기울었다는 뜻 (0.956인데 틀린 승객 있음), 실제 빈도는 보정으로 확인</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">재현율 ↑ · 정밀도 ↓, FN ↓ · FP ↑</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">규칙을 먼저 선언 → validation에서 고르기 → 고른 기준 하나로 test 1회</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">아님. 양성 예측이 적은 구간에서 내려갈 수 있음 (0.65 · 0.95)</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">생존자 · 사망자 한 명씩 뽑으면 생존자 확률이 더 높을 확률이 0.857</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">PR-AUC (ROC-AUC는 양성이 드물어도 0.95 근처로 유지)</td>
</tr>
<tr>
<td style="text-align: left;">6-1</td>
<td style="text-align: left;">시그모이드 전의 logit</td>
</tr>
<tr>
<td style="text-align: left;">6-2</td>
<td style="text-align: left;"><code>class_weight="balanced"</code> · <code>pos_weight</code> (효과는 재현율 · 정밀도 · AP로 확인)</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">모델은 train 교차검증, 기준은 valid, 최종 성적은 test 1회</td>
</tr>
</tbody>
</table>

#### 8-2 답

```python
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. 혼동행렬 숫자 → 지표 4개 (공식)
tn, fp, fn, tp = 50, 10, 5, 35
prec, rec = tp / (tp + fp), tp / (tp + fn)
print(round((tp + tn) / (tn + fp + fn + tp), 3))  # 0.85
print(round(prec, 3), round(rec, 3))  # 0.778 0.875
print(round(2 * prec * rec / (prec + rec), 3))  # 0.824

# 2. 기준 0.5 → 0.3
y = np.array([0, 1, 0, 1, 1, 0, 1, 0])
p = np.array([0.2, 0.45, 0.6, 0.8, 0.35, 0.1, 0.9, 0.4])
y5, y3 = (p >= 0.5).astype(int), (p >= 0.3).astype(int)
print(round(precision_score(y, y5), 3), recall_score(y, y5))  # 0.667 0.5
print(round(precision_score(y, y3), 3), recall_score(y, y3))  # 0.667 1.0

# 3. AUC = (양성 점수 > 음성 점수)인 쌍의 비율
pos, neg = p[y == 1], p[y == 0]
pair = (pos[:, None] > neg[None, :]).mean()
print(pair, roc_auc_score(y, p))  # 0.8125 0.8125

# 4. 성별 × 아이(12세 이하) 생존률
df = pd.read_csv("train.csv")
age = df.dropna(subset=["Age"]).copy()
age["아이"] = age["Age"] <= 12
rate = age.groupby(["Sex", "아이"])["Survived"].mean().round(3)
print(rate.tolist())  # [0.777, 0.594, 0.173, 0.568]

# 5. Sex 하나 vs Pclass 하나 → valid AUC
X_dev, _, y_dev, _ = train_test_split(
    df, df["Survived"], test_size=0.2, stratify=df["Survived"],
    random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
sex_prep = ColumnTransformer([("c", OneHotEncoder(), ["Sex"])])
cls_prep = ColumnTransformer([("c", StandardScaler(), ["Pclass"])])
aucs = []
for prep in [sex_prep, cls_prep]:
    m = Pipeline([("prep", prep), ("lr", LogisticRegression(max_iter=5000))])
    m.fit(X_tr, y_tr)
    aucs.append(round(roc_auc_score(y_va, m.predict_proba(X_va)[:, 1]), 3))
print(aucs)  # [0.78, 0.702]
```

**4번 해석** · 남자는 아이(0.568)가 성인(0.173)보다 3배 이상 높지만, 여자는 아이(0.594)가 성인(0.777)보다 낮음 → '아이 먼저'는 남자 안에서만 뚜렷. 숫자 하나로 넣은 `Age`로는 이런 성별별 차이를 직선 하나로 표현하기 어려움. **5번 해석** · Sex 하나(0.78)가 Pclass 하나(0.702)보다 높고, 0장의 전체 특성 로지스틱은 0.857 (5-1)

---
## 9. 면접 30초 답변

**Q1. 정확도만으로 모델을 평가하면 안 되는 이유는?**
> 정확도는 맞힌 개수만 세기 때문에 어떤 실수를 했는지 알 수 없습니다. 양성이 10%인 데이터에서 전부 음성으로 찍어도 90%가 나오고, 타이타닉에서는 '남자면 사망' 규칙 한 줄이 학습한 로지스틱과 정확도가 같았습니다. 그래서 혼동행렬로 실수를 나누고, 업무에서 비싼 실수에 맞춰 정밀도나 재현율, F1을 함께 봅니다.

**Q2. 정밀도와 재현율의 차이와 고르는 기준은?**
> 정밀도는 양성이라고 예측한 것 중 진짜 양성의 비율이라 오탐을, 재현율은 실제 양성 중 찾아낸 비율이라 놓침을 봅니다. 스팸 차단처럼 잘못 울리는 비용이 크면 정밀도, 암 진단이나 사기 탐지처럼 놓치는 비용이 크면 재현율을 우선합니다. 둘은 임곗값으로 서로 교환되므로 업무 규칙을 먼저 정합니다.

**Q3. 임곗값은 어떻게 정하나요?**
> 모델이 낸 확률에 기본 0.5 대신 업무 기준을 적용합니다. F1 최대, 재현율 0.9 이상 중 정밀도 최대처럼 규칙을 먼저 선언하고 검증 데이터에서 고른 뒤, 고른 기준 하나로 테스트를 한 번만 평가합니다. 테스트 결과를 보고 기준을 다시 바꾸면 테스트가 검증이 되어 성능이 부풀려집니다.

**Q4. ROC-AUC와 PR-AUC는 언제 쓰나요?**
> ROC-AUC는 모든 임곗값에서 양성을 음성보다 위에 세우는 능력으로, 클래스가 비교적 고를 때 모델 비교에 씁니다. 양성이 드문 데이터에서는 음성이 많아 오탐이 늘어도 ROC가 거의 변하지 않으므로, 정밀도를 직접 반영하는 PR-AUC를 봅니다. 실제로 양성을 1%대로 줄이면 ROC-AUC는 0.95로 유지되는데 AP는 0.35까지 떨어졌습니다.

**Q5. 신경망 이진 분류의 출력층과 손실은 어떻게 구성하나요?**
> 마지막 Linear 층이 점수인 logit 하나를 내고, 학습에는 그 logit을 그대로 받는 BCEWithLogitsLoss를 씁니다. 이것은 로지스틱 회귀의 시그모이드와 교차 엔트로피와 같은 구조이고, 예측할 때만 시그모이드로 확률을 만들어 임곗값을 적용합니다. 불균형하면 pos_weight로 양성 실수에 큰 벌점을 줍니다.

---
## 10. 전체 코드

1 ~ 7장 정답 코드를 한 셀에 모음. **이 셀만 실행해도 전체가 동작**하며(`train.csv`는 같은 폴더), 각 출력 앞 `[번호]`가 섹션.

In [21]:
# ---------- 0. 준비 ----------
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score, log_loss)

SEED = 42
titanic = pd.read_csv("train.csv")
X_t = titanic.drop(columns="Survived").copy()
X_t["Cabin1"] = X_t["Cabin"].fillna("N").str[:1]
X_t = X_t.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
y_t = titanic["Survived"]
X_dev, X_test, y_dev, y_test = train_test_split(
    X_t, y_t, test_size=0.2, stratify=y_t, random_state=SEED)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)
NUM = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CAT = ["Sex", "Embarked", "Cabin1"]


def make_lr(cols=NUM + CAT, C=1.0):
    num = Pipeline([("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())])
    cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    prep = ColumnTransformer([
        ("num", num, [c for c in cols if c in NUM]),
        ("cat", cat, [c for c in cols if c in CAT])])
    return Pipeline([("prep", prep),
                     ("model", LogisticRegression(max_iter=5000, C=C))])


digits = load_digits()
X_dig, y7 = digits.data, (digits.target == 7).astype(int)
print("[0]", len(X_train), len(X_valid), len(X_test), round(y7.mean(), 3))


# ---------- 1. 정확도의 함정 ----------
lr_all = make_lr().fit(X_train, y_train)
rule = (X_valid["Sex"] == "female").astype(int)
print("[1-1]", round(accuracy_score(y_valid, rule), 3),
      round(lr_all.score(X_valid, y_valid), 3))

X7_tr, X7_te, y7_tr, y7_te = train_test_split(
    X_dig, y7, test_size=0.2, stratify=y7, random_state=11)
fake = np.zeros(len(y7_te), dtype=int)
print("[1-2]", accuracy_score(y7_te, fake),
      recall_score(y7_te, fake, zero_division=0))


# ---------- 2. 혼동행렬과 지표 ----------
pred7 = LogisticRegression(max_iter=5000).fit(X7_tr, y7_tr).predict(X7_te)
print("[2-1]", confusion_matrix(y7_te, pred7).ravel())
print("[2-2]", round(precision_score(y7_te, pred7), 4),
      round(recall_score(y7_te, pred7), 4), round(f1_score(y7_te, pred7), 4))


# ---------- 3. 로지스틱 회귀와 확률 ----------
proba = lr_all.predict_proba(X_valid)[:, 1]
z_val = lr_all.decision_function(X_valid)
print("[3-2]", np.allclose(1 / (1 + np.exp(-z_val)), proba))
print("[3-3]", round(log_loss(y_valid, proba), 4))
lr_ps = make_lr(["Pclass", "Sex"]).fit(X_train, y_train)
combo = pd.DataFrame({"Pclass": [3, 1], "Sex": ["female", "male"]})
print("[3-4]", lr_ps.predict_proba(combo)[:, 1].round(3))


# ---------- 4. 임곗값 ----------
for thr in [0.3, 0.5, 0.7]:
    y_hat = (proba >= thr).astype(int)
    print("[4-1]", thr, round(precision_score(y_valid, y_hat), 3),
          round(recall_score(y_valid, y_hat), 3))

X7_dev, X7_test, y7_dev, y7_test = train_test_split(
    X_dig, y7, test_size=0.2, stratify=y7, random_state=42)
X7_fit, X7_val, y7_fit, y7_val = train_test_split(
    X7_dev, y7_dev, test_size=0.25, stratify=y7_dev, random_state=42)
pipe7 = Pipeline([("pca", PCA(n_components=10, random_state=42)),
                  ("lr", LogisticRegression(max_iter=5000, C=0.1))])
p_val = pipe7.fit(X7_fit, y7_fit).predict_proba(X7_val)[:, 1]
thrs = np.round(np.arange(0.1, 1.0, 0.1), 1)
t_f1 = thrs[int(np.argmax([f1_score(y7_val, p_val >= t) for t in thrs]))]
p_test = pipe7.predict_proba(X7_test)[:, 1] >= t_f1
print("[4-2]", t_f1, round(f1_score(y7_test, p_test), 3))


# ---------- 5. 곡선 지표 ----------
print("[5-1]", round(roc_auc_score(y_valid, proba), 3),
      round(roc_auc_score(y7_val, p_val), 3))
weak = Pipeline([("pca", PCA(n_components=3, random_state=42)),
                 ("lr", LogisticRegression(max_iter=5000, C=0.1))])
p_weak = weak.fit(X7_fit, y7_fit).predict_proba(X7_val)[:, 1]
print("[5-2]", round(roc_auc_score(y7_val, p_weak), 3),
      round(average_precision_score(y7_val, p_weak), 3))


# ---------- 6. 딥러닝 연결 ----------
logits = torch.tensor(z_val, dtype=torch.float32)
target = torch.tensor(y_valid.to_numpy(), dtype=torch.float32)
bce = torch.nn.BCEWithLogitsLoss()(logits, target).item()
print("[6-1]", round(bce, 4))
pos_w = (y7_fit == 0).sum() / (y7_fit == 1).sum()
print("[6-2]", round(float(pos_w), 2))


# ---------- 7. 종합 실습: 타이타닉 전체 흐름 ----------
final = make_lr(C=0.1).fit(X_train, y_train)
p_v = final.predict_proba(X_valid)[:, 1]
grid_t = np.round(np.arange(0.1, 0.91, 0.05), 2)
best_t = grid_t[int(np.argmax([f1_score(y_valid, p_v >= t)
                               for t in grid_t]))]
p_t = final.predict_proba(X_test)[:, 1]
print("[7]", best_t, round(f1_score(y_test, p_t >= best_t), 3),
      round(roc_auc_score(y_test, p_t), 3))

[0]

 534 178 179 0.1
[1-1] 0.798 0.798
[1-2] 0.9 0.0
[2-1] [324   0   2  34]
[2-2] 1.0 0.9444 0.9714
[3-2] True
[3-3] 0.446
[3-4] [0.592 0.436]
[4-1] 0.3 0.671 0.838
[4-1] 0.5 0.729 0.75
[4-1] 0.7 0.814 0.515


[4-2] 0.3 0.932
[5-1] 0.857 0.997


[5-2] 0.961 0.701
[6-1] 0.446
[6-2] 9.07
[7] 0.35 0.711 0.835


---
## 11. 전체 최종 요약표

### 등급별 핵심

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">기준선</td>
<td style="text-align: left;">정확도는 다수 클래스 · 규칙 한 줄과 나란히</td>
<td style="text-align: left;">첫 epoch 전 기준선 계산</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">불균형</td>
<td style="text-align: left;">Positive를 먼저 정하고 재현율 · 정밀도</td>
<td style="text-align: left;">이상 탐지 · 의료 영상</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">혼동행렬</td>
<td style="text-align: left;">행 = 실제 · 열 = 예측, <code>tn, fp, fn, tp</code></td>
<td style="text-align: left;">다중 분류 혼동행렬</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">정밀도 · 재현율 · F1</td>
<td style="text-align: left;">오탐 · 놓침 · 균형</td>
<td style="text-align: left;">epoch마다 F1 기록</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">시그모이드</td>
<td style="text-align: left;">z = 0 ⇔ 확률 0.5</td>
<td style="text-align: left;">이진 분류 출력층</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">predict_proba</td>
<td style="text-align: left;">기준을 정하려면 확률</td>
<td style="text-align: left;"><code>torch.sigmoid(model(x))</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">교차 엔트로피</td>
<td style="text-align: left;">확신하고 틀릴수록 큰 벌점, 확률 ≠ 정답 보장</td>
<td style="text-align: left;">학습 손실</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">임곗값</td>
<td style="text-align: left;">낮추면 재현율 ↑ · 정밀도 ↓</td>
<td style="text-align: left;"><code>prob &gt;= thr</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">기준 선택</td>
<td style="text-align: left;">규칙 먼저 → validation에서 → test 1회</td>
<td style="text-align: left;"><code>best.pt</code>의 val 확률로</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">ROC-AUC</td>
<td style="text-align: left;">기준 무관 순위 능력, 0.5 = 찍기</td>
<td style="text-align: left;">의료 · 이상 탐지 지표</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">PR-AUC</td>
<td style="text-align: left;">불균형에서 양성 쪽 성능, 기준선 = 양성 비율</td>
<td style="text-align: left;">객체 탐지 mAP</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">BCEWithLogits</td>
<td style="text-align: left;">logit을 넣음 (= log loss)</td>
<td style="text-align: left;">이진 분류 손실</td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">불균형 가중치</td>
<td style="text-align: left;"><code>class_weight</code> · <code>pos_weight</code></td>
<td style="text-align: left;"><code>pos_weight=음성/양성</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">종합</td>
<td style="text-align: left;">모델 = CV, 기준 = valid, 성적 = test 1회</td>
<td style="text-align: left;">같은 5단계</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">macro · weighted</td>
<td style="text-align: left;">소수 클래스는 macro</td>
<td style="text-align: left;">macro F1</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">계수</td>
<td style="text-align: left;">부호로 방향, 크기 비교는 조심</td>
<td style="text-align: left;">설명용 기준선 모델</td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">줄다리기 예외</td>
<td style="text-align: left;">양성 예측이 적으면 흔들림</td>
<td style="text-align: left;">기준은 0.05 ~ 0.1 간격</td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Positive와 업무 실수 비용을 먼저 적기</td>
<td style="text-align: left;">정확도 하나로 결론</td>
</tr>
<tr>
<td style="text-align: left;">혼동행렬부터 보기</td>
<td style="text-align: left;"><code>precision_score(예측, 정답)</code> 순서 실수</td>
</tr>
<tr>
<td style="text-align: left;">기준 선택 규칙을 표 보기 전에 선언</td>
<td style="text-align: left;">test로 기준 · 모델 고르기</td>
</tr>
<tr>
<td style="text-align: left;">불균형이면 PR-AUC · 재현율</td>
<td style="text-align: left;">확률 0.97을 '97% 맞음'으로 해석</td>
</tr>
<tr>
<td style="text-align: left;">고른 기준 하나로 test 1회</td>
<td style="text-align: left;">모델 끝 sigmoid + <code>BCEWithLogitsLoss</code></td>
</tr>
<tr>
<td style="text-align: left;">test가 낮아도 그대로 보고</td>
<td style="text-align: left;">test를 보고 0.5로 되돌리기</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도 vs 정밀도 vs 재현율 vs F1</td>
<td style="text-align: left;">전체 vs 오탐 기준 vs 놓침 기준 vs 둘의 균형</td>
<td style="text-align: left;">업무의 실수 비용으로 고름</td>
</tr>
<tr>
<td style="text-align: left;">FP vs FN</td>
<td style="text-align: left;">음성을 양성이라 함(오탐) vs 양성을 놓침</td>
<td style="text-align: left;">어느 쪽이 비싼지 먼저 적기</td>
</tr>
<tr>
<td style="text-align: left;">macro vs weighted</td>
<td style="text-align: left;">클래스 같은 비중 vs 샘플 수 비중</td>
<td style="text-align: left;">소수 클래스가 중요하면 macro</td>
</tr>
<tr>
<td style="text-align: left;"><code>predict</code> vs <code>predict_proba</code></td>
<td style="text-align: left;">0/1 vs 확률</td>
<td style="text-align: left;">기준을 정하려면 proba</td>
</tr>
<tr>
<td style="text-align: left;">점수 z vs 확률</td>
<td style="text-align: left;">-∞ ~ +∞ vs 0 ~ 1 (z = 0 ⇔ 0.5)</td>
<td style="text-align: left;">손실은 z, 설명은 확률</td>
</tr>
<tr>
<td style="text-align: left;">정확도 vs log loss</td>
<td style="text-align: left;">맞음 · 틀림 vs 확신까지 채점</td>
<td style="text-align: left;">확률 품질은 log loss</td>
</tr>
<tr>
<td style="text-align: left;">기준 0.5 vs 업무 기준</td>
<td style="text-align: left;">모델 기본 vs 비용에 맞춘 선택</td>
<td style="text-align: left;">validation에서 업무 기준</td>
</tr>
<tr>
<td style="text-align: left;">validation vs test (기준 선택)</td>
<td style="text-align: left;">여러 번 보며 고름 vs 마지막 1번</td>
<td style="text-align: left;">test로 고르지 않음</td>
</tr>
<tr>
<td style="text-align: left;">ROC-AUC vs PR-AUC</td>
<td style="text-align: left;">음성 많으면 둔감 vs 양성 드물수록 엄격</td>
<td style="text-align: left;">불균형이면 PR-AUC</td>
</tr>
<tr>
<td style="text-align: left;">AUC vs F1</td>
<td style="text-align: left;">기준 무관 순위 vs 한 기준의 판정</td>
<td style="text-align: left;">비교는 AUC, 보고는 기준과 함께</td>
</tr>
<tr>
<td style="text-align: left;"><code>BCEWithLogitsLoss</code> vs <code>BCELoss</code></td>
<td style="text-align: left;">logit 입력 vs 확률 입력</td>
<td style="text-align: left;">WithLogits</td>
</tr>
<tr>
<td style="text-align: left;"><code>class_weight</code> vs 기준 낮추기</td>
<td style="text-align: left;">학습 단계 vs 판정 단계</td>
<td style="text-align: left;">둘 다 재현율을 올림</td>
</tr>
</tbody>
</table>